# Ya-Risk — Bilan V21 — Qwen3.8 / pré-certification analyste

> Base : `bilan_v20c_7.ipynb` (V13c interne), enrichie pour le workflow Ya-Risk.
>
> **Objectif V21 :** 1 PDF → JSON brut immuable → contrôles Ya-Risk → JSON avant certification → classeur `MODEL ETAT FINANCIER - ANALYSTE COPILOTE V20.xlsx` pour comparaison PDF ↔ Excel.
>
> **V22 (revue) :** tolérance écarts portée à **±1000 DZD** (±1 KDZD) partout ; ajout de la
> **Partie 3 — transposition multi-exercices** (N → N-4) avec reset, filet débit/crédit,
> corrections validées et Checklist complète (reprise du flux Copilot validé).
>
> Principes :
> - Qwen **recopie** les montants imprimés ; il ne déplace jamais une valeur Débit/Crédit pour la rendre « logique ».
> - `null` signifie **information absente / non lue**, jamais zéro.
> - les cellules Excel contenant une formule ne sont **jamais écrasées** ;
> - les montants imprimés sur les lignes de formule servent uniquement au **contrôle** ;
> - toute anomalie TCR (sens inhabituel, total non lu) reste visible dans le JSON, `0.Données extraites` et la `Checklist`.
> - la certification humaine n’est pas encore écrite dans ce notebook : V21 produit l’état **A_CERTIFIER**.


In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 1 — DEPENDANCES | BILANS_V13
# Objectif: installer le runtime minimal.
# ════════════════════════════════════════════════════════════
%pip install -q -U 'transformers>=4.57.0' accelerate pymupdf pillow psutil
print('✅ Dependances OK')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 2 — IMPORTS | YARISK_V21
# ════════════════════════════════════════════════════════════
import time, json, re, gc, copy, unicodedata, shutil, subprocess, sys, os
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import Counter
import fitz, torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
import openpyxl
from openpyxl.comments import Comment
from openpyxl.styles import Alignment, Border, Font, PatternFill, Protection, Side
print('✅ Imports OK')


In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 3 — CONFIG | YARISK_V21 — Qwen3.8 / H100
# ════════════════════════════════════════════════════════════
MODEL_NAME = os.environ.get('QWEN_MODEL_NAME', 'Qwen3.8-27B-FP8')
MODEL_PATH = os.environ.get(
    'QWEN_MODEL_PATH',
    '/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.8-27B-FP8/main'
)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

MAX_NEW_TOKENS = 4096
IMAGE_MAX_SIZE = 2024
MIN_PIXELS = 4 * 32 * 32
MAX_PIXELS = 2000 * 32 * 32
PDF_ZOOM = 3.0
BLANK_THRESHOLD = 0.95
CLASSIF_BATCH_SIZE = 32
GPU_BATCH_SIZE = 8
PDF_WORKERS = 4

INPUT_DIR = Path('/mnt/Risk/bilans_in')
OUTPUT_DIR = Path('/mnt/Risk/bilans_out')
JSON_DIR = OUTPUT_DIR / 'json_bruts_v21'
LOG_PATH = OUTPUT_DIR / 'pipeline_bilans_v21.log'
INDEX_PATH = OUTPUT_DIR / 'index_bilans_v21.jsonl'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)
pdfs = sorted(INPUT_DIR.glob('*.pdf'))

# ── H100 CUDA tuning ──
if DEVICE == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')

print('Modele : ' + MODEL_NAME)
print('Chemin : ' + MODEL_PATH)
print('Device : ' + DEVICE + ' | PDF: ' + str(len(pdfs)) + ' | JSON: ' + str(JSON_DIR))
print('Batch sizes — classif: ' + str(CLASSIF_BATCH_SIZE) +
      ' | extraction: ' + str(GPU_BATCH_SIZE) +
      ' | PDF workers: ' + str(PDF_WORKERS))


In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 4 — LOG | YARISK_V21
# ════════════════════════════════════════════════════════════
def log(msg):
    ligne = datetime.now().strftime('%Y-%m-%d %H:%M:%S') + ' — ' + str(msg)
    print(ligne, flush=True)
    with open(LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(ligne + chr(10))
print('✅ Log OK')


In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 5 — CHARGEMENT MODELE | vLLM 0.27.1 + CUDA 13 + H100
# ════════════════════════════════════════════════════════════
import os, time, glob, shutil

# ── 1. Localiser nvcc (installé par cuda-toolkit dans site-packages) ──
_nvcc = None
for _p in glob.glob('/opt/conda/envs/DWS-GPU/lib/python3.11/site-packages/nvidia/cu13/bin/nvcc'):
    _nvcc = _p
    break
if _nvcc is None:
    _hits = glob.glob('/opt/conda/envs/**/nvidia/cu*/bin/nvcc', recursive=True)
    _nvcc = _hits[0] if _hits else None

if _nvcc:
    _cuda_home = os.path.dirname(os.path.dirname(_nvcc))   # .../nvidia/cu13
    os.environ['CUDA_HOME'] = _cuda_home
    os.environ['CUDA_PATH'] = _cuda_home
    os.environ['PATH'] = os.path.dirname(_nvcc) + os.pathsep + os.environ.get('PATH', '')
    print('CUDA_HOME =', _cuda_home)
    print('nvcc      =', shutil.which('nvcc') or 'INTROUVABLE dans PATH')
else:
    print('⚠️ nvcc introuvable — DeepGEMM sera désactivé')
    os.environ['VLLM_USE_DEEP_GEMM'] = '0'

# ── 2. Désactiver FlashInfer sampler (Python 3.11 incompatible) ──
os.environ['VLLM_USE_FLASHINFER_SAMPLER']  = '0'
os.environ['VLLM_ATTENTION_BACKEND']       = 'FLASH_ATTN'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'

from vllm import LLM, SamplingParams
from transformers import AutoProcessor

t0 = time.time()

processor = AutoProcessor.from_pretrained(
    MODEL_PATH, trust_remote_code=True,
    min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS,
)
processor.tokenizer.padding_side = 'left'

llm = LLM(
    model=MODEL_PATH,
    dtype='bfloat16',
    quantization='fp8',
    gpu_memory_utilization=0.92,
    max_model_len=8192,
    max_num_seqs=16,
    trust_remote_code=True,
    enforce_eager=False,
    enable_prefix_caching=True,
    limit_mm_per_prompt={'image': 1},
    mm_processor_kwargs={'min_pixels': MIN_PIXELS, 'max_pixels': MAX_PIXELS},
)

SAMPLING = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_NEW_TOKENS,
    repetition_penalty=1.0,
)

print('✅ ' + MODEL_NAME + ' charge en ' + str(round(time.time()-t0, 1)) + 's | FP8 | FA3 | PagedAttention')


In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 6 — UTILITAIRES + INFERENCE vLLM | H100
# ════════════════════════════════════════════════════════════
from concurrent.futures import ThreadPoolExecutor
import unicodedata, re, json, gc
import numpy as np
import fitz
from PIL import Image

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def resize(img, max_side=IMAGE_MAX_SIZE):
    w, h = img.size
    if max(w, h) <= max_side: return img
    r = max_side / max(w, h)
    return img.resize((int(w*r), int(h*r)), Image.LANCZOS)

def strip_accents(s):
    s = str(s)
    return ''.join(c for c in unicodedata.normalize('NFKD', s)
                   if not unicodedata.combining(c))

def norm_key(s):
    s = strip_accents(str(s)).lower()
    s = re.sub('[^a-z0-9]+', ' ', s)
    return s.strip()

def estimate_skew(img):
    small = img.convert('L').copy(); small.thumbnail((500, 500))
    def score(a):
        r = np.array(small.rotate(a, expand=True, fillcolor=255)) < 128
        proj = r.sum(axis=1)
        return float((proj ** 2).sum())
    best = max(range(-12, 13, 2), key=score)
    best = max([best-1, best-0.5, best, best+0.5, best+1], key=score)
    return best if abs(best) >= 1 else 0.0

def deskew(img):
    a = estimate_skew(img)
    if a: img = img.rotate(a, expand=True, fillcolor=(255,255,255),
                            resample=Image.BICUBIC)
    return img, a

def is_blank(image, threshold=BLANK_THRESHOLD):
    arr = np.array(image.convert('L'))
    return (arr > 240).sum() / arr.size >= threshold

def _render_one_page(args):
    pdf_path, page_idx, zoom, max_side = args
    doc = fitz.open(pdf_path)
    matrix = fitz.Matrix(zoom, zoom)
    pix = doc.load_page(page_idx).get_pixmap(matrix=matrix, alpha=False)
    img = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
    doc.close()
    img, angle = deskew(img)
    img = resize(img, max_side)
    return {'index': page_idx, 'image': img,
            'largeur_px': img.width, 'hauteur_px': img.height,
            'rotation_estimee': angle}

def pdf_to_pages(path, zoom=PDF_ZOOM):
    """Rendu PDF parallélisé sur CPU."""
    doc = fitz.open(path); n = len(doc); doc.close()
    if n <= 2 or PDF_WORKERS <= 1:
        doc = fitz.open(path); matrix = fitz.Matrix(zoom, zoom); pages = []
        for i in range(len(doc)):
            pix = doc.load_page(i).get_pixmap(matrix=matrix, alpha=False)
            img = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
            img, angle = deskew(img); img = resize(img)
            pages.append({'index': i, 'image': img,
                          'largeur_px': img.width, 'hauteur_px': img.height,
                          'rotation_estimee': angle})
        doc.close(); return pages
    args = [(str(path), i, zoom, IMAGE_MAX_SIZE) for i in range(n)]
    with ThreadPoolExecutor(max_workers=PDF_WORKERS) as pool:
        pages = list(pool.map(_render_one_page, args))
    pages.sort(key=lambda p: p['index'])
    return pages

def parse_json(text):
    if not text: return {}
    text = text.strip()
    try: return json.loads(text)
    except Exception: pass
    start = text.find('{'); end = text.rfind('}')
    if start >= 0 and end > start:
        try: return json.loads(text[start:end+1])
        except Exception: return {}
    return {}

def apply_template(messages):
    try:
        return processor.apply_chat_template(
            messages, tokenize=False,
            add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)

# ── Inference vLLM ──────────────────────────────────────────
# vLLM gère en interne : continuous batching, PagedAttention,
# KV cache, FP8, CUDA graphs.
# ask_batch envoie TOUTES les images en un seul appel llm.generate()
# → le scheduler vLLM les parallélise automatiquement.

def _build_inputs(prompt, images):
    """Construit la liste de dicts multimodaux pour llm.generate()."""
    inputs = []
    for img in images:
        msgs = [{'role': 'user', 'content': [
            {'type': 'image', 'image': img},
            {'type': 'text',  'text': prompt},
        ]}]
        text = apply_template(msgs)
        inputs.append({'prompt': text, 'multi_modal_data': {'image': img}})
    return inputs

def ask_single(prompt, image):
    t0 = time.time()
    inputs = _build_inputs(prompt, [image])
    outputs = llm.generate(inputs, SAMPLING)
    o = outputs[0]
    return {
        'text':       o.outputs[0].text,
        'tokens_in':  len(o.prompt_token_ids),
        'tokens_out': len(o.outputs[0].token_ids),
        'elapsed':    round(time.time() - t0, 2),
    }

def ask_batch(prompt, images):
    """
    Envoie toutes les images en un seul llm.generate().
    vLLM fait du continuous batching : dès qu'une séquence finit,
    le slot est réutilisé — pas d'attente du plus lent comme avec
    transformers.generate() en batch naïf.
    """
    if not images: return []
    t0 = time.time()
    inputs = _build_inputs(prompt, images)
    outputs = llm.generate(inputs, SAMPLING)
    el = time.time() - t0
    results = []
    for o in outputs:
        results.append({
            'text':       o.outputs[0].text,
            'tokens_in':  len(o.prompt_token_ids),
            'tokens_out': len(o.outputs[0].token_ids),
            'elapsed':    round(el / len(images), 2),
        })
    return results

print('✅ Utilitaires OK | inference vLLM (continuous batching)')
print('   ask_batch → llm.generate() unique pour N images')


In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 7 — SCHEMAS + CLASSIFICATION CONTENU + PROMPTS | YARISK_V21
# (identique V12 : signatures texte, en-tete par page)
# ════════════════════════════════════════════════════════════
SCHEMAS = {
 'ACTIF': {'titre': 'BILAN (ACTIF)', 'cols': ['montant_brut', 'amortissements_provisions_pertes', 'net_n', 'net_n1'], 'postes': {
   'ecarts_acquisition_goodwill': 'Ecart d acquisition goodwill', 'immobilisations_incorporelles': 'Immobilisations incorporelles',
   'terrains': 'Terrains', 'batiments': 'Batiments', 'autres_immobilisations_corporelles': 'Autres Immobilisations corporelles',
   'immobilisations_corporelles': 'Immobilisations corporelles',
   'immobilisations_en_concession': 'Immobilisations en concession', 'immobilisations_en_cours': 'Immobilisations en cours',
   'titres_mis_en_equivalence': 'Titres mis en equivalence', 'autres_participations_creances': 'Autres participations et creances rattachees',
   'autres_titres_immobilises': 'Autres titres immobilises', 'prets_actifs_financiers_non_courants': 'Prets et autres actifs financiers non courants',
   'impots_differes_actif': 'Impots Differes Actif',
   'immobilisations_financieres': 'Immobilisations financieres', 'total_actif_non_courant': 'TOTAL ACTIF NON COURANT',
   'stocks_encours': 'Stocks et encours', 'clients': 'Clients', 'autres_debiteurs': 'Autres debiteurs',
   'impots_assimiles_actif': 'Impots et assimiles', 'autres_creances_assimiles': 'Autres Creances et Emplois assimiles',
   'creances_emplois_assimiles': 'Creances et emplois assimiles',
   'disponibilites_assimiles': 'Disponibilites et assimiles',
   'placements_financiers_courants': 'Placements et autres actifs financiers courants', 'tresorerie_actif': 'Tresorerie',
   'total_actif_courant': 'TOTAL ACTIF COURANT', 'total_general_actif': 'TOTAL GENERAL ACTIF'}},
 'PASSIF': {'titre': 'BILAN (PASSIF)', 'cols': ['n', 'n1'], 'postes': {
   'capital_emis': 'Capital emis', 'capital_non_appele': 'Capital non appele', 'primes_reserves': 'Primes et reserves',
   'ecart_reevaluation': 'Ecart de reevaluation', 'ecart_equivalence': 'Ecart d equivalence', 'resultat_net_passif': 'Resultat net',
   'report_a_nouveau': 'Report a nouveau', 'part_societe_consolidante': 'Part de la societe consolidante', 'part_minoritaires': 'Part des minoritaires',
   'total_capitaux_propres': 'TOTAL I', 'emprunts_dettes_financieres': 'Emprunts et dettes financieres',
   'impots_differes_provisionnes': 'Impots differes et provisionnes', 'autres_dettes_non_courantes': 'Autres dettes non courantes',
   'provisions_produits_avance': 'Provisions et produits constatés d avance', 'total_passifs_non_courants': 'TOTAL PASSIFS NON COURANTS II',
   'fournisseurs_rattaches': 'Fournisseurs et comptes rattaches', 'impots_passif': 'Impots', 'autres_dettes': 'Autres dettes',
   'tresorerie_passif': 'Tresorerie Passif', 'total_passifs_courants': 'TOTAL PASSIFS COURANTS', 'total_general_passif': 'TOTAL GENERAL PASSIF'}},
 'TCR': {'titre': 'COMPTE DE RESULTAT', 'cols': ['n_debit', 'n_credit', 'n1_debit', 'n1_credit'], 'postes': {
   'ventes_marchandises': 'Ventes de Marchandises', 'produits_fabriques': 'Produits Fabriques', 'prestations_services': 'Prestations de Services',
   'ventes_travaux': 'Ventes de Travaux', 'produits_annexes': 'Produits Annexes', 'rabais_remises_ristournes_accordes': 'Rabais remises ristournes accordes',
   'chiffre_affaires_net': 'Chiffre d affaires net', 'production_stockee_destockee': 'Production Stockee ou destockee',
   'production_immobilisee': 'Production immobilisee', 'subvention_exploitation': 'Subvention d exploitation', 'production_exercice': 'I-Production de l exercice',
   'achats_marchandises_vendues': 'Achats de Marchandises vendues', 'matieres_premieres': 'Matieres premieres', 'autres_approvisionnements': 'Autres Approvisionnements',
   'variation_stocks': 'Variation des Stocks', 'achats_etudes_prestations': 'Achats d Etudes et de Prestations de services', 'rabais_remises_ristournes_obtenus_achats': 'Rabais remises ristournes obtenus sur achats',
   'autres_consommations': 'Autres consommations',
   'sous_traitance_generale': 'Sous-traitance generale', 'locations': 'Locations', 'entretien_reparations': 'Entretien reparations et maintenance',
   'primes_assurances': 'Primes d assurances', 'personnel_exterieur': 'Personnel exterieur a l entreprise', 'remuneration_intermediaires': 'Remuneration d intermediaires et honoraires',
   'publicite': 'Publicite', 'deplacements_missions': 'Deplacements missions et receptions', 'rabais_remises_ristournes_obtenus_services': 'Rabais remises ristournes obtenus sur services exterieurs',
   'autres_services': 'Autres services',
   'consommations_exercice': 'II-Consommations de l exercice', 'valeur_ajoutee_exploitation': 'III-Valeur ajoutee d exploitation',
   'charges_personnel': 'Charges de personnel', 'impots_taxes_assimiles': 'Impots et taxes et versements assimiles', 'excedent_brut_exploitation': 'IV-Excedent brut d exploitation',
   'autres_produits_operationnels': 'Autres produits operationnels', 'autres_charges_operationnelles': 'Autres charges operationnelles',
   'dotations_amortissements': 'Dotations aux amortissements', 'provisions': 'Provisions', 'pertes_valeur': 'Perte de Valeur',
   'reprises_pertes_valeur_provisions': 'Reprise sur pertes de valeur et provisions', 'resultat_operationnel': 'V-Resultat operationnel',
   'produits_financiers': 'Produits financiers', 'charges_financieres': 'Charges financieres', 'resultat_financier': 'VI-Resultat Financier',
   'resultat_ordinaire': 'VII-Resultat ordinaire', 'elements_extraordinaires_produits': 'Elements extraordinaires Produits',
   'elements_extraordinaires_charges': 'Elements extraordinaires Charges', 'resultat_extraordinaire': 'VIII-Resultat extraordinaire',
   'impots_exigibles_resultats': 'Impots exigibles sur resultats', 'impots_differes_resultats': 'Impots differes sur resultats', 'resultat_net_exercice': 'RESULTAT NET DE L EXERCICE'}},
 'DECL': {'titre': 'DECLARATION IBS', 'cols': ['valeur'], 'kinds': {'nif': 'nif', 'raison_sociale': 'texte', 'activite_principale': 'texte', 'registre_commerce': 'texte', 'adresse_siege': 'texte', 'cac_cabinet': 'texte', 'cac_nom': 'texte', 'exercice_annee': 'annee', 'annee_souscription': 'annee'}, 'postes': {
   'nif': 'Numero d Identification Fiscale', 'raison_sociale': 'Designation de l entreprise', 'activite_principale': 'Activite principale',
   'registre_commerce': 'Registre de Commerce', 'adresse_siege': 'Adresse siege social', 'cac_cabinet': 'Certification des comptes - Cabinet',
   'cac_nom': 'Certification des comptes - Nom CAC', 'exercice_annee': 'Resultat de l exercice - Annee', 'annee_souscription': 'Annee de souscription',
   'chiffre_affaires_global_ht': 'Chiffre d affaires global hors taxes', 'resultat_comptable': 'Resultat comptable', 'resultat_fiscal': 'Resultat fiscal'}},
 'A1': {'titre': '1/ Tableau des mouvements des stocks', 'cols': ['solde_debut', 'debit', 'credit', 'solde_fin'], 'postes': {
   'stocks_marchandises': 'Stocks de marchandises', 'matieres_fournitures': 'Matieres et fournitures', 'autres_approvisionnements': 'Autres approvisionnements',
   'encours_production_biens': 'Encours de production de biens', 'encours_production_services': 'Encours de production de services', 'stocks_produits': 'Stocks de produits',
   'stocks_provenant_immobilisations': 'Stocks provenant d immobilisations', 'stocks_exterieur': 'Stocks a l exterieur', 'total': 'TOTAL'}},
 'A2': {'titre': '2/ Tableau de la fluctuation de la production stockee', 'cols': ['debit', 'credit', 'solde_debiteur', 'solde_crediteur'], 'postes': {
   'production_stockee': 'Production stockee ou destockee (ligne unique si non libellee)'}},
 'A3': {'titre': '3/ Charges de personnel, impots, taxes, autres services', 'cols': ['montant'], 'postes': {
   'charges_locatives': 'Charges locatives et charges de copropriete', 'etudes_recherches': 'Etudes et recherches', 'documentation_divers': 'Documentation et divers',
   'transports_biens': 'Transports de biens et transport collectif du personnel', 'frais_postaux': 'Frais postaux et de telecommunications',
   'services_bancaires': 'Services bancaires et assimiles', 'cotisations_divers': 'Cotisations et divers', 'total_autres_services': 'TOTAL (1)',
   'remunerations_personnel': 'Remunerations du personnel', 'remuneration_exploitant': 'Remunerations de l exploitant individuel (cas d une EURL)',
   'cotisations_sociales': 'Cotisations aux organismes sociaux', 'charges_sociales_exploitant': 'Charges sociales de l exploitant individuel (cas d une EURL)',
   'autres_charges_sociales': 'Autres charges sociales', 'autres_charges_personnel': 'Autres charges de personnels', 'total_charges_personnel': 'TOTAL (2)',
   'impots_sur_remunerations': 'Impots taxes et versements assimiles sur remunerations', 'impots_non_recuperables': 'Impots et taxes non recuperables sur chiffres d affaires',
   'autres_impots_taxes': 'Autres impots et taxes (hors impots sur les resultats)', 'total_impots': 'TOTAL (3)', 'total_general': 'TOTAL (1)+(2)+(3)'}},
 'A4': {'titre': '4/ Autres charges et produits operationnels', 'cols': ['montant'], 'postes': {
   'redevances_concessions_charges': 'Redevances pour concessions brevets licences logiciels (charges)', 'moins_values_sorties_actifs': 'Moins values sur sorties d actifs immobilises non financiers',
   'jetons_presence_charges': 'Jetons de presence (charges)', 'pertes_creances_irrecouvrables': 'Perte sur creances irrecouvrables',
   'quote_part_operations_commun_charges': 'Quote-part de resultat sur operations faites en commun (charges)', 'amendes_penalites_dons': 'Amendes et penalites subventions accordees dons et liberalites',
   'charges_exceptionnelles_gestion': 'Charges exceptionnelles de gestion courante', 'autres_charges_gestion': 'Autres charges de gestion courante', 'total_charges': 'TOTAL (charges)',
   'redevances_concessions_produits': 'Redevances pour concessions brevets licences logiciels (produits)', 'plus_values_sorties_actifs': 'Plus values sur sorties d actifs immobilises non financiers',
   'jetons_presence_produits': 'Jetons de presence et remunerations d administrateurs ou de gerant', 'quotes_parts_subventions_virees': 'Quotes-parts de subventions d investissement virees au resultat',
   'quote_part_operations_commun_produits': 'Quote-part de resultat sur operations faites en commun (produits)', 'rentrees_creances_amorties': 'Rentree sur creances amorties',
   'produits_exceptionnels_gestion': 'Produits exceptionnels sur operations de gestion', 'autres_produits_gestion': 'Autres produits de gestion courante', 'total_produits': 'TOTAL (produits)'}},
 'A5': {'titre': '5/ Tableau des amortissements et pertes de valeurs', 'cols': ['dotations_cumulees_debut', 'dotations_exercice', 'diminutions_elements_sortis', 'dotations_cumulees_fin', 'dotations_fiscales_exercice', 'ecarts'], 'postes': {
   'goodwill': 'Goodwill', 'immobilisations_incorporelles': 'Immobilisations incorporelles', 'immobilisations_corporelles': 'Immobilisations corporelles',
   'participations': 'Participations', 'autres_actifs_financiers_non_courants': 'Autres actifs financiers non courants', 'total': 'TOTAL'}},
 'A6': {'titre': '6/ Tableau des immobilisations creees ou acquises', 'cols': ['montants_bruts', 'tva_deduite', 'montant_net_a_amortir'], 'postes': {
   'goodwill': 'Goodwill', 'immobilisations_incorporelles': 'Immobilisations incorporelles', 'immobilisations_corporelles': 'Immobilisations corporelles',
   'participations': 'Participations', 'autres_actifs_financiers_non_courants': 'Autres actifs financiers non courants', 'total': 'TOTAL'}},
 'A7': {'titre': '7/ Tableau des immobilisations cedees', 'dynamic': True, 'str_cols': ['date_acquisition'], 'cols': ['date_acquisition', 'montant_net_actif', 'amortissements_pratiques', 'valeur_nette_comptable', 'prix_cession', 'plus_value', 'moins_value'], 'postes': {}},
 'A8': {'titre': '8/ Tableau des provisions et pertes de valeurs', 'cols': ['provisions_cumulees_debut', 'dotations_exercice', 'reprises_exercice', 'provisions_cumulees_fin'], 'postes': {
   'pertes_valeur_stocks': 'Pertes de valeurs sur stocks', 'pertes_valeur_creances': 'Pertes de valeurs sur creances', 'pertes_valeur_actions': 'Pertes de valeurs sur actions et parts sociales',
   'provisions_pensions': 'Provisions pour pensions et obligations similaires', 'provisions_litiges': 'Provisions sur litiges', 'autres_provisions_personnel': 'Autres provisions liees au personnel',
   'provisions_impots': 'Provisions pour impots', 'autres_provisions': 'Autres provisions', 'total': 'TOTAL'}},
 'A81': {'titre': '8/1 Releve des pertes de valeurs sur creances', 'dynamic': True, 'cols': ['valeur_creance', 'perte_valeur_constituee'], 'postes': {}},
 'A82': {'titre': '8/2 Releve des pertes de valeurs sur actions', 'dynamic': True, 'cols': ['valeur_nominale_debut', 'perte_valeur_constituee', 'valeur_nette_comptable'], 'postes': {}},
 'A9': {'titre': '9/ Tableau de determination du resultat fiscal', 'cols': ['montant'], 'postes': {
   'resultat_net_benefice': 'I. Resultat net de l exercice Benefice', 'resultat_net_perte': 'I. Resultat net de l exercice Perte',
   'charges_immeubles_non_affectes': 'Charges des immeubles non affectes directement a l exploitation', 'quote_part_cadeaux_publicitaires': 'Quote-part des cadeaux publicitaires non deductibles',
   'quote_part_sponsoring': 'Quote-part du sponsoring et parrainage non deductibles', 'frais_reception': 'Frais de reception non deductibles', 'cotisations_dons': 'Cotisations et dons non deductibles',
   'impots_taxes_non_deductibles': 'Impots et taxes non deductibles', 'provisions_non_deductibles': 'Provisions non deductibles', 'amortissements_non_deductibles': 'Amortissements non deductibles',
   'quote_part_frais_rd': 'Quote-part des frais de recherche developpement non deductibles', 'amortissements_credit_bail_preneur': 'Amortissements non deductibles credit bail (Preneur)',
   'loyers_hors_produits_financiers_bailleur': 'Loyers hors produits financiers (bailleur)', 'ibs_impot_exigible': 'Impots sur les benefices - Impot exigible sur le resultat',
   'ibs_impot_differe': 'Impots sur les benefices - Impot differe (variation)', 'pertes_valeur_non_deductibles': 'Pertes de valeurs non deductibles', 'amendes_penalites': 'Amendes et penalites',
   'autres_reintegrations': 'Autres reintegrations', 'total_reintegrations': 'Total des reintegrations', 'plus_values_cession_actif_immobilise': 'Plus values sur cession d elements d actif immobilises',
   'produits_plus_values_actions_bourse': 'Produits et plus values de cession des actions et titres assimiles cotes en bourse', 'revenus_distribution_benefices': 'Revenus provenant de la distribution des benefices',
   'amortissements_credit_bail_bailleur': 'Amortissements credit bail (Bailleur)', 'loyers_hors_charges_financieres_preneur': 'Loyers hors charges financieres (Preneur)',
   'complement_amortissements': 'Complement d amortissements', 'autres_deductions': 'Autres deductions', 'total_deductions': 'Total des deductions',
   'total_deficits_a_deduire': 'Total des deficits a deduire', 'resultat_fiscal_benefice': 'Resultat fiscal (I+II-III-IV) Benefice', 'resultat_fiscal_deficit': 'Resultat fiscal (I+II-III-IV) Deficit'}},
 'A10': {'titre': '10/ Tableau d affectation du resultat et des reserves (N-1)', 'cols': ['montant'], 'postes': {
   'origine_report_a_nouveau_n1': 'Report a nouveau de l exercice N-1', 'origine_resultat_n1': 'Resultat de l exercice N-1', 'origine_prelevements_reserves': 'Prelevements sur reserves',
   'origine_total': 'TOTAL (origine)', 'affectation_reserves': 'Reserves', 'affectation_augmentation_capital': 'Augmentation du capital', 'affectation_dividendes': 'Dividendes',
   'affectation_report_a_nouveau': 'Report a nouveau', 'affectation_total': 'TOTAL (affectation)'}},
 'A11': {'titre': '11/ Tableau des participations', 'dynamic': True, 'cols': ['capitaux_propres', 'dont_capital', 'quote_part_capital_pct', 'resultat_dernier_exercice', 'prets_avances', 'dividendes_encaisses', 'valeur_comptable_titres'], 'postes': {}},
 'A12': {'titre': '12/ Commissions courtages redevances honoraires sous-traitance', 'dynamic': True, 'str_cols': ['nif', 'adresse'], 'cols': ['nif', 'adresse', 'montant_percu'], 'postes': {}},
 'A13': {'titre': '13/ Taxe sur l activite professionnelle', 'dynamic': True, 'cols': ['ca_imposable', 'ca_exonere', 'tap_acquittee'], 'postes': {}},
 'ST': {'titre': 'Operations de sous-traitance', 'dynamic': True, 'str_cols': ['nif', 'article', 'adresse', 'reference_contrat'], 'cols': ['nif', 'article', 'adresse', 'reference_contrat', 'montant'], 'postes': {}},
 'REM': {'titre': 'Remunerations versees aux membres de certaines societes', 'dynamic': True, 'str_cols': ['annee_versement'], 'cols': ['nombre_parts', 'annee_versement', 'traitement_emoluments', 'representation_mission_forfait', 'representation_mission_remboursement', 'frais_pro_forfait', 'frais_pro_remboursement'], 'postes': {}},
 'DIST': {'titre': 'Repartition des produits des actions et parts sociales distribues', 'cols': ['montant'], 'postes': {
   'montant_global_brut': 'Montant global brut des distributions', 'paye_par_societe': 'Paye par la societe elle meme',
   'paye_par_etablissement': 'Paye par un etablissement charge du service des titres', 'total_revenus_repartis': 'Montant total des revenus repartis'}}
}

# ── Référentiel canonique TCR Ya-Risk ──────────────────────
# C = produit / crédit habituel ; D = charge / débit habituel ;
# B = les deux côtés sont possibles selon le signe / la nature du solde.
TCR_SENS = {
    'ventes_marchandises':'C', 'produits_fabriques':'C', 'prestations_services':'C',
    'ventes_travaux':'C', 'produits_annexes':'C', 'production_immobilisee':'C',
    'subvention_exploitation':'C', 'rabais_remises_ristournes_obtenus_achats':'C',
    'rabais_remises_ristournes_obtenus_services':'C', 'reprises_pertes_valeur_provisions':'C',
    'autres_produits_operationnels':'C', 'produits_financiers':'C',
    'elements_extraordinaires_produits':'C',

    'rabais_remises_ristournes_accordes':'D', 'achats_marchandises_vendues':'D',
    'matieres_premieres':'D', 'autres_approvisionnements':'D',
    'achats_etudes_prestations':'D', 'autres_consommations':'D',
    'sous_traitance_generale':'D', 'locations':'D', 'entretien_reparations':'D',
    'primes_assurances':'D', 'personnel_exterieur':'D',
    'remuneration_intermediaires':'D', 'publicite':'D',
    'deplacements_missions':'D', 'autres_services':'D',
    'charges_personnel':'D', 'impots_taxes_assimiles':'D',
    'autres_charges_operationnelles':'D', 'dotations_amortissements':'D',
    'provisions':'D', 'pertes_valeur':'D', 'charges_financieres':'D',
    'elements_extraordinaires_charges':'D', 'impots_exigibles_resultats':'D',
    'impots_differes_resultats':'D',

    'production_stockee_destockee':'B', 'variation_stocks':'B',
    'chiffre_affaires_net':'B', 'production_exercice':'B',
    'consommations_exercice':'B', 'valeur_ajoutee_exploitation':'B',
    'excedent_brut_exploitation':'B', 'resultat_operationnel':'B',
    'resultat_financier':'B', 'resultat_ordinaire':'B',
    'resultat_extraordinaire':'B', 'resultat_net_exercice':'B'
}

TCR_CONTROLE_FORMULE = {
    'chiffre_affaires_net', 'production_exercice', 'consommations_exercice',
    'valeur_ajoutee_exploitation', 'excedent_brut_exploitation',
    'resultat_operationnel', 'resultat_financier', 'resultat_ordinaire',
    'resultat_extraordinaire', 'resultat_net_exercice'
}

TCR_TOTAUX_ATTENDUS = list(TCR_CONTROLE_FORMULE)

def role_tcr(row_code):
    return 'CONTROLE_FORMULE' if row_code in TCR_CONTROLE_FORMULE else 'SAISIE'

ANNEXE_NUM_MAP = {1:'A1', 2:'A2', 3:'A3', 4:'A4', 5:'A5', 6:'A6', 7:'A7', 8:'A8', 9:'A9', 10:'A10', 11:'A11', 12:'A12', 13:'A13'}

TITLE_SIGNATURES = [
    ('A81', ['RELEVE DES PERTES DE VALEURS SUR CREANC'], []),
    ('A82', ['RELEVE DES PERTES DE VALEURS SUR ACTIONS'], []),
    ('A1',  ['MOUVEMENTS DES STOCKS'], []),
    ('A2',  ['FLUCTUATION DE LA PRODUCTION STOCKEE'], []),
    ('A3',  ['CHARGES DE PERSONNEL', 'VERSEMENTS ASSIMILES'], []),
    ('A4',  ['AUTRES CHARGES ET PRODUITS OPERATIONNELS'], []),
    ('A5',  ['AMORTISSEMENTS ET PERTES DE VALEURS'], []),
    ('A6',  ['IMMOBILISATIONS CREEES OU ACQUISES'], []),
    ('A7',  ['IMMOBILISATIONS CEDEES'], []),
    ('A8',  ['PROVISIONS ET PERTES DE VALEURS'], ['RELEVE DES PERTES']),
    ('A9',  ['DETERMINATION DU RESULTAT FISCAL'], []),
    ('A10', ['AFFECTATION DU RESULTAT ET DES RESERVES'], []),
    ('A11', ['TABLEAU DES PARTICIPATIONS'], []),
    ('A12', ['COMMISSIONS ET COURTAGES'], []),
    ('A13', ['TAXE SUR L ACTIVITE PROFESSIONNELLE'], []),
    ('ST',  ['OPERATIONS DE SOUS-TRAITANCE'], ['COMMISSIONS ET COURTAGES']),
    ('REM', ['REMUNERATIONS VERSEES AUX MEMBRES'], []),
    ('DIST',['REPARTITION DES PRODUITS DES ACTIONS'], []),
]
# Signatures TCR suite (2e page) : priment sur les annexes car ces
# libelles n'apparaissent que dans le TCR, jamais dans les annexes.
TCR_SUITE_SIGNATURES = [
    ('TCR', ['RESULTAT OPERATIONNEL'], ['DETERMINATION DU RESULTAT FISCAL']),
    ('TCR', ['RESULTAT FINANCIER'], ['DETERMINATION DU RESULTAT FISCAL']),
    ('TCR', ['RESULTAT NET DE L EXERCICE'], ['DETERMINATION DU RESULTAT FISCAL']),
    ('TCR', ['RESULTAT ORDINAIRE'], ['DETERMINATION DU RESULTAT FISCAL']),
    ('TCR', ['EXCEDENT BRUT'], ['DETERMINATION DU RESULTAT FISCAL']),
    ('TCR', ['VALEUR AJOUTEE'], ['DETERMINATION DU RESULTAT FISCAL']),
]

MAIN_SIGNATURES = [
    ('ACTIF',  ['BILAN', 'ACTIF'],  ['PASSIF']),
    ('PASSIF', ['BILAN', 'PASSIF'], []),
    ('TCR',    ['COMPTE DE RESULTAT'], []),
    ('DECL',   ['DECLARATION'], []),
]
def _norm(t):
    t = strip_accents(t or '').upper()
    t = t.replace(chr(39), ' ').replace(chr(8217), ' ')
    t = re.sub(r'[^A-Z0-9/ ]+', ' ', t)
    return ' '.join(t.split())
_TITLE_SIGS = [(c, [_norm(m) for m in ms], [_norm(f) for f in fs]) for c, ms, fs in TITLE_SIGNATURES]
_TCR_SUITE_SIGS = [(c, [_norm(m) for m in ms], [_norm(f) for f in fs])
                   for c, ms, fs in TCR_SUITE_SIGNATURES]
_MAIN_SIGS = [(c, [_norm(m) for m in ms], [_norm(f) for f in fs]) for c, ms, fs in MAIN_SIGNATURES]
def _match_signatures(t, signatures):
    found = []
    for code, musts, forbids in signatures:
        if all(m in t for m in musts) and not any(f in t for f in forbids):
            if code not in found: found.append(code)
    return found
def _num_from_segment(t):
    codes, nums = [], []
    for m in re.finditer(r'(?:^| )(1[0-3]|[0-9])/([0-9])?(?![0-9])', t):
        n = int(m.group(1)); s = m.group(2)
        code = ('A8' + s) if (n == 8 and s in ('1', '2')) else ANNEXE_NUM_MAP.get(n)
        if code and code not in codes:
            codes.append(code); nums.append(n)
    return codes, nums
def types_from_title(titre):
    brut = _norm(titre)
    if not brut: return [], []
    segments = [_norm(s) for s in re.split(r'\s\|\s', titre or '') if s.strip()]
    segments = [s for s in segments if s] or [brut]
    types, nums = [], []
    for seg in segments:
        # TCR suite : soldes intermediaires propres au TCR (2e page)
        par_tcr_suite = _match_signatures(seg, _TCR_SUITE_SIGS)
        par_texte = _match_signatures(seg, _TITLE_SIGS)
        if par_tcr_suite:
            for c in par_tcr_suite:
                if c not in types: types.append(c)
        elif par_texte:
            for c in par_texte:
                if c not in types: types.append(c)
        else:
            codes, ns = _num_from_segment(seg)
            for c in codes:
                if c not in types: types.append(c)
            nums.extend(ns)
    if ('A81' in types or 'A82' in types) and 'A8' in types:
        if not any('PROVISIONS ET PERTES DE VALEURS' in s for s in segments): types.remove('A8')
    if not types:
        types = _match_signatures(brut, _MAIN_SIGS)
        if 'ACTIF' in types and 'PASSIF' in types:
            types = ['ACTIF'] if brut.find('ACTIF') < brut.find('PASSIF') else ['PASSIF']
    return types, nums

PROMPT_CLASSIF = chr(10).join([
 'Lis cette page scannee d une liasse fiscale algerienne (imprime Serie G).',
 'Ta seule tache : recopier les TITRES DE TABLEAUX imprimes sur cette page.',
 'QU EST-CE QU UN TITRE : au-dessus d un tableau, souvent cadre/souligne, commence par 1/ 2/ ... 13/ 8/1 8/2, ou titre central BILAN (ACTIF)/(PASSIF), COMPTE DE RESULTAT, DECLARATION...',
 'CE QUI N EST PAS UN TITRE : libelles de lignes, en-tetes de colonnes, mentions N.I.F/Designation/Exercice, IMPRIME DESTINE A L ADMINISTRATION.',
 'ATTENTION : 9/ contient une ligne interne I. Resultat net de l exercice (Compte de resultat) : c est un LIBELLE, pas un titre.',
 'Le COMPTE DE RESULTAT (TCR) occupe souvent DEUX pages : la 1re porte le titre COMPTE DE RESULTAT, la 2e continue avec V-Resultat operationnel et se termine par IX-Resultat net. Les deux pages sont de type TCR, pas ANNEXE.',
 'UNE PAGE PEUT CONTENIR DEUX TABLEAUX : regarde HAUT et BAS. Recopie TOUS les titres, meme coupes.',
 'RECOPIE LE LIBELLE EN ENTIER, mot pour mot, avec son numero.',
 'Titres de reference : 1/ mouvements des stocks ; 2/ fluctuation production stockee ; 3/ charges personnel impots taxes autres services ; 4/ autres charges et produits operationnels ; 5/ amortissements et pertes de valeurs ; 6/ immobilisations creees ou acquises ; 7/ immobilisations cedees ; 8/ provisions et pertes de valeurs ; 8/1 releve pertes creances ; 8/2 releve pertes actions ; 9/ determination du resultat fiscal ; 10/ affectation resultat et reserves (N-1) ; 11/ participations ; 12/ commissions courtages redevances honoraires sous-traitance ; 13/ taxe activite professionnelle.',
 'CAS PARTICULIER — PAGE SANS TITRE DE TABLEAU :',
 'Si la page ne porte AUCUN titre (ni numero N/ ni titre central encadre),',
 'identifie le tableau par son CONTENU :',
 '',
 '  SUITE DU TCR (type TCR) :',
 '  La page contient des lignes en gras numerotees en chiffres romains :',
 '    V-Resultat operationnel',
 '    VI-Resultat financier',
 '    VII-Resultat ordinaire (V+VI)',
 '    VIII-Resultat extraordinaire',
 '    IX-RESULTAT NET DE L EXERCICE',
 '  et des lignes de detail comme : Autres produits operationnels,',
 '  Dotations aux amortissements, Produits financiers, Charges financieres,',
 '  Impots exigibles sur resultats, Impots differes sur resultats.',
 '  Les en-tetes de colonnes (Debit/Credit) ne sont PAS repetes sur cette page',
 '  (ils etaient sur la page precedente du TCR).',
 '  -> Repondre: {"titre": "COMPTE DE RESULTAT (suite)", "type": "TCR"}',
 '',
 '  TABLEAU 9 (type ANNEXE) — A NE PAS CONFONDRE :',
 '  Le tableau 9 se reconnait par :',
 '  - un titre encadre : 9/ Tableau de determination du resultat fiscal',
 '  - une structure en 4 sections : I. Resultat net (Benefice/Perte),',
 '    II. Reintegrations (liste de charges non deductibles),',
 '    III. Deductions, IV. Deficits anterieurs a deduire',
 '  - une seule colonne de montants',
 '  - se termine par Resultat fiscal (I+II-III-IV) Benefice / Deficit',
 '  -> C est une ANNEXE, pas la suite du TCR.',
 '',
 'Reponds en JSON strict: {"titre": "<titres separes par |>", "type": "<ACTIF|PASSIF|TCR|DECL|ANNEXE|AUTRE>"}'])

RULES = [
 'REGLES: JSON valide uniquement, sans markdown, sans backticks.',
 'Aucune valeur inventee. Case vide ou absente: null. Illisible: null. IMPORTANT: null signifie absence d information, jamais zero.',
 'Montants en nombres JSON sans separateurs de milliers.',
 'Montant entre parentheses = negatif: (1 553 799) devient -1553799.',
 'Textes (nif, adresses, noms, dates) en chaines telles quelles.']

def build_prompt(types):
    L = []
    L.append('Lis cette page scannee d une liasse fiscale algerienne (imprime Serie G).')
    L.append('Extrais en JSON strict les tableaux suivants, identifies par leur code.')
    L.append('Structure: { type_page, numero_page_imprimee, titre_page, entete: {entreprise, nif, exercice, exercice_du, exercice_au, adresse, activite, serie_g}, puis une cle par code de tableau present.')
    # Le TCR peut occuper 2 pages : la 2e n a pas de titre mais le meme schema
    if 'TCR' in types:
        L.append('Le COMPTE DE RESULTAT peut occuper 2 pages. Si cette page est '
                 'la suite du TCR (pas de titre, lignes Resultat operationnel a Resultat net), '
                 'extrais les postes normalement avec le code TCR.')
        L.append('REGLES TCR — PRIORITAIRES :')
        L.append('1. Pour chaque exercice, identifie d abord les EN-TETES imprimes DEBIT et CREDIT. '
                 'Ne deduis jamais la colonne uniquement de sa position visuelle.')
        L.append('2. Recopie le montant EXACTEMENT dans la colonne ou il est imprime. '
                 'Ne deplace JAMAIS un montant pour le rendre coherent avec la nature du poste.')
        L.append('3. Une case vide reste null. Ne remplace jamais null par 0.')
        L.append('4. Les produits sont habituellement au CREDIT et les charges au DEBIT. '
                 'Cette information est un garde-fou de lecture, PAS une autorisation de deplacer une valeur.')
        L.append('5. Les soldes et totaux (I a IX, chiffre d affaires net, resultats) doivent etre LUS tels qu imprimes. '
                 'Ne les recalcule pas : Ya-Risk les comparera ensuite aux formules.')
        L.append('6. Sur la 2e page sans en-tetes, conserve strictement l alignement des colonnes du tableau. '
                 'Utilise les libelles V, VI, VII, VIII, IX comme ancres; en cas de doute persistant: null.')
        L.append('7. Ne confonds jamais la suite du TCR avec le tableau 9 de determination du resultat fiscal.')
    L.append('EN-TETE — obligatoire sur CHAQUE page :')
    L.append('- nif : 15 chiffres colles, sans barre ni espace; recopie ce qui est lisible.')
    L.append('- entreprise : texte apres Designation de l entreprise, forme juridique comprise.')
    L.append('- exercice : date apres Exercice clos le (jj/mm/aaaa); si Exercice du..au.., remplis exercice_du/exercice_au et date de fin dans exercice.')
    for t in types:
        spec = SCHEMAS[t]
        L.append('--- Code ' + t + ' : ' + spec['titre'] + ' ---')
        if spec.get('dynamic'):
            L.append('Tableau a lignes libres. Retourne: {lignes: [ {libelle_imprime, valeurs: {colonne: nombre ou texte ou null}} ]}')
            L.append('Colonnes: ' + ', '.join(spec['cols']))
            L.append('Seulement les lignes non vides; garde les totaux si presents.')
        else:
            L.append('Retourne: {lignes: [ {row_code, libelle_imprime, valeurs: {colonne: nombre ou null}} ]}')
            L.append('Colonnes: ' + ', '.join(spec['cols']))
            if t == 'DECL': L.append('Les valeurs peuvent etre des nombres ou des textes selon le poste.')
            L.append('Row_code autorises:')
            for k, lab in spec['postes'].items():
                if t == 'TCR':
                    sens = {'C':'CREDIT', 'D':'DEBIT', 'B':'DEBIT_OU_CREDIT'}.get(TCR_SENS.get(k), 'DEBIT_OU_CREDIT')
                    L.append('- ' + k + ' : ' + lab + ' | mode=' + role_tcr(k) + ' | sens_habituel=' + sens)
                else:
                    L.append('- ' + k + ' : ' + lab)
            L.append('Retourne seulement les row_code avec au moins une valeur non nulle.')
    L.extend(RULES)
    return chr(10).join(L)
VALID_CODES = set(SCHEMAS.keys()) | {'AUTRE', 'BLANCHE'}
print('✅ Schemas + classification + referentiel TCR Ya-Risk V21 OK')


In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 8 — NORMALISATION | YARISK_V21
# ════════════════════════════════════════════════════════════
def norm_str(v):
    if v is None: return None
    if isinstance(v, bool): return None
    s = ' '.join(str(v).split())
    return s if s and s.lower() not in ('null','none','n/a','na','-') else None
def norm_montant(v):
    if v is None: return None
    if isinstance(v, bool): return None
    if isinstance(v, (int, float)): return float(v)
    s = str(v).strip()
    if s.lower() in ('null','none','n/a','na','-'): return None
    neg = (s.startswith('(') and s.endswith(')')) or s.startswith('-')
    s = re.sub('[^0-9.,-]', '', s)
    if not s: return None
    if s.count(',') == 1 and '.' not in s: s = s.replace(',', '.')
    elif ',' in s: s = s.replace(',', '')
    elif s.count('.') > 1: s = s.replace('.', '')
    try: return -float(s) if neg else float(s)
    except Exception: return None
def norm_nif(v):
    s = norm_str(v)
    return re.sub('[^0-9]', '', s) if s else None
def norm_annee4(v):
    s = norm_str(v) or ''
    m = re.findall('20[0-9]{2}', s)
    return m[-1] if m else None
def norm_by_kind(v, kind):
    if kind == 'texte': return norm_str(v)
    if kind == 'nif': return norm_nif(v)
    if kind == 'annee': return norm_annee4(v)
    return norm_montant(v)
def normalise_fixed(t, block):
    spec = SCHEMAS[t]; kinds = spec.get('kinds', {})
    lignes = (block or {}).get('lignes') or []
    label_to_key = {norm_key(v): k for k, v in spec['postes'].items()}
    row_map = {}
    for lg in lignes:
        if not isinstance(lg, dict): continue
        rc = norm_str(lg.get('row_code'))
        if rc not in spec['postes']:
            lab = norm_str(lg.get('libelle_imprime'))
            if lab and norm_key(lab) in label_to_key: rc = label_to_key[norm_key(lab)]
        if rc in spec['postes']: row_map[rc] = lg
    out = {}
    for key in spec['postes']:
        vals = (row_map.get(key) or {}).get('valeurs') or {}
        if not isinstance(vals, dict): vals = {}
        out[key] = {}
        for col in spec['cols']:
            out[key][col] = norm_by_kind(vals.get(col), kinds.get(key, 'montant'))
    return out
def normalise_dynamic(t, block):
    spec = SCHEMAS[t]; str_cols = set(spec.get('str_cols', []))
    out = []
    for lg in (block or {}).get('lignes') or []:
        if not isinstance(lg, dict): continue
        vals = lg.get('valeurs') or {}
        if not isinstance(vals, dict): vals = {}
        row = {'libelle_imprime': norm_str(lg.get('libelle_imprime'))}
        for col in spec['cols']:
            row[col] = norm_str(vals.get(col)) if col in str_cols else norm_montant(vals.get(col))
        if row['libelle_imprime'] or any(v is not None for k, v in row.items() if k != 'libelle_imprime'): out.append(row)
    return out
def normalise_table(t, block):
    if t == 'AUTRE': return block or {}
    if SCHEMAS[t].get('dynamic'): return normalise_dynamic(t, block)
    return normalise_fixed(t, block)
def normalise_entete(data):
    ent = data.get('entete') or {}
    if not isinstance(ent, dict): ent = {}
    return {'entreprise': norm_str(ent.get('entreprise')), 'nif': norm_nif(ent.get('nif')), 'exercice': norm_str(ent.get('exercice')),
            'exercice_du': norm_str(ent.get('exercice_du')), 'exercice_au': norm_str(ent.get('exercice_au')), 'adresse': norm_str(ent.get('adresse')),
            'activite': norm_str(ent.get('activite')), 'serie_g': norm_str(ent.get('serie_g'))}
def count_values(obj):
    count = 0
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == 'brut': continue
            count += count_values(v)
    elif isinstance(obj, list):
        for v in obj: count += count_values(v)
    elif isinstance(obj, (int, float)) and not isinstance(obj, bool): count += 1
    elif isinstance(obj, str) and obj: count += 1
    return count
print('✅ Normalisation OK')


In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 9 — CONSTRUCTION PAGES JSON | YARISK_V21
# ════════════════════════════════════════════════════════════
def build_blank_page(page, fichier_source):
    n = page['index'] + 1
    return {'page_id': 'p' + str(n).zfill(3), 'pdf_page_index': page['index'], 'numero_page_scannee': n, 'numero_page_imprimee': None, 'fichier_source': fichier_source,
            'classification': {'type_page': 'BLANCHE', 'types': [], 'annexe_numeros': [], 'sous_type_page': 'PAGE_VIERGE', 'titre_page': None, 'page_annexe': False, 'page_utile': False, 'page_blanche': True},
            'image': {'largeur_px': page.get('largeur_px'), 'hauteur_px': page.get('hauteur_px'), 'rotation_estimee': page.get('rotation_estimee'), 'deskew_applique': bool(page.get('rotation_estimee'))},
            'statut_extraction': {'statut': 'BLANCHE', 'nb_tableaux': 0, 'nb_champs_extraits': 0, 'commentaire': 'Page blanche detectee, non envoyee au VLM.'},
            'entete_page': {}, 'donnees': {}, 'tokens_in': 0, 'tokens_out': 0, 'temps_s': 0.0}
def build_page_object(page, types, data, rep, fichier_source):
    n = page['index'] + 1
    data = data if isinstance(data, dict) else {}
    donnees = {'brut': data}
    for t in types:
        if t == 'AUTRE': donnees['AUTRE'] = {'tables': data.get('tables') or []}
        else: donnees[t] = normalise_table(t, data.get(t))
    statut = 'OK' if data else 'ECHEC_EXTRACTION'
    nums = page.get('annexe_nums') or []
    return {'page_id': 'p' + str(n).zfill(3), 'pdf_page_index': page['index'], 'numero_page_scannee': n,
            'numero_page_imprimee': data.get('numero_page_imprimee') if isinstance(data.get('numero_page_imprimee'), int) else None,
            'fichier_source': fichier_source,
            'classification': {'type_page': types[0] if len(types) == 1 else (types[0] if types else 'AUTRE'), 'types': types, 'annexe_numeros': nums,
                               'sous_type_page': ' '.join(types) if types else 'AUTRE', 'titre_page': norm_str(data.get('titre_page')),
                               'page_annexe': any(t.startswith('A') or t in ('ST','REM','DIST') for t in types), 'page_utile': bool(types), 'page_blanche': False},
            'image': {'largeur_px': page.get('largeur_px'), 'hauteur_px': page.get('hauteur_px'), 'rotation_estimee': page.get('rotation_estimee'), 'deskew_applique': bool(page.get('rotation_estimee'))},
            'statut_extraction': {'statut': statut, 'nb_tableaux': len(types), 'nb_champs_extraits': count_values(donnees), 'commentaire': None},
            'entete_page': normalise_entete(data), 'donnees': donnees,
            'tokens_in': rep.get('tokens_in') if rep else 0, 'tokens_out': rep.get('tokens_out') if rep else 0, 'temps_s': rep.get('elapsed') if rep else 0.0}
def merge_core(base, new):
    if new is None: return base
    if base is None: return copy.deepcopy(new)
    for key, cols in new.items():
        if key not in base: base[key] = copy.deepcopy(cols)
        elif isinstance(cols, dict):
            for col, val in cols.items():
                if base[key].get(col) is None and val is not None: base[key][col] = val
    return base
def build_synthese(page_objects):
    synthese = {'actif': None, 'passif': None, 'tcr': None, 'decl': None, 'annexes': {}, 'annexes_pages': []}
    for page in page_objects:
        types = page['classification'].get('types') or []
        donnees = page.get('donnees', {})
        if 'ACTIF' in types and synthese['actif'] is None: synthese['actif'] = donnees.get('ACTIF')
        if 'PASSIF' in types and synthese['passif'] is None: synthese['passif'] = donnees.get('PASSIF')
        if 'TCR' in types: synthese['tcr'] = merge_core(synthese['tcr'], donnees.get('TCR'))
        if 'DECL' in types and synthese['decl'] is None: synthese['decl'] = donnees.get('DECL')
        for t in types:
            if t in ('ACTIF','PASSIF','TCR','DECL','AUTRE'): continue
            if synthese['annexes'].get(t) is None: synthese['annexes'][t] = donnees.get(t)
        if any(t not in ('ACTIF','PASSIF','TCR','DECL','AUTRE') for t in types):
            synthese['annexes_pages'].append({'page_id': page['page_id'], 'numero_page_scannee': page['numero_page_scannee'], 'types': types, 'annexe_numeros': page['classification'].get('annexe_numeros')})
    return synthese
def build_document(pdf_path, pages, page_objects, tok_in, tok_out, elapsed):
    nb_bl = sum(1 for p in page_objects if p['classification']['type_page'] == 'BLANCHE')
    nb_ut = sum(1 for p in page_objects if p['classification']['type_page'] != 'BLANCHE')
    nb_an = sum(1 for p in page_objects if p['classification'].get('page_annexe'))
    return {'schema_version': '21.0', 'type_document': 'liasse_fiscale_algerienne_serie_g',
            'document': {'document_id': 'doc_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + pdf_path.stem, 'fichier_source': pdf_path.name,
                         'nb_pages_pdf': len(pages), 'nb_pages_scannes': len(pages), 'nb_pages_blanches': nb_bl, 'nb_pages_utiles': nb_ut, 'nb_pages_annexes': nb_an,
                         'date_extraction': datetime.now().strftime('%Y-%m-%d %H:%M:%S'), 'duree_extraction_s': round(elapsed, 2),
                         'modele_extraction': MODEL_NAME, 'modele_path': MODEL_PATH, 'prompt_version': 'v21', 'referentiel_version': 'liasse_serie_g_v21',
                         'tokens_in': tok_in, 'tokens_out': tok_out, 'tokens_total': tok_in + tok_out},
            'pages': page_objects, 'synthese': build_synthese(page_objects)}
print('✅ Construction pages JSON OK')


In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 10 — PIPELINE PRINCIPAL | YARISK_V21
# ════════════════════════════════════════════════════════════
deja = {f.stem for f in JSON_DIR.glob('*.json')}
a_traiter = [p for p in pdfs if p.stem not in deja]
log('A traiter: ' + str(len(a_traiter)) + ' | deja traites: ' + str(len(deja)))
t_total = time.time(); n_ok = 0; n_err = 0; index_records = []; prompt_cache = {}
# ── Warmup GPU avant le premier PDF ──
_warmup_gpu()

for num, pdf_path in enumerate(a_traiter, start=1):
    prefix = '[' + str(num).zfill(4) + '/' + str(len(a_traiter)) + '] '
    t_pdf = time.time()
    log(prefix + 'DEMARRAGE ' + pdf_path.name)
    try:
        t0 = time.time()
        pages = pdf_to_pages(pdf_path)
        log(prefix + 'rendu termine: ' + str(len(pages)) + ' pages en ' + str(round(time.time()-t0, 1)) + 's')
        page_objects = []; active_pages = []
        for p in pages:
            if is_blank(p['image']): page_objects.append(build_blank_page(p, pdf_path.name))
            else: active_pages.append(p)
        log(prefix + 'pages blanches: ' + str(len(page_objects)) + ' | pages actives: ' + str(len(active_pages)))
        tok_in = 0; tok_out = 0
        if active_pages:
            t0 = time.time()
            mini = [resize(p['image'], 600) for p in active_pages]
            reps1 = []
            for bs in chunks(mini, CLASSIF_BATCH_SIZE): reps1 += ask_batch(PROMPT_CLASSIF, bs)
            tok_in += sum(r['tokens_in'] for r in reps1); tok_out += sum(r['tokens_out'] for r in reps1)
            log(prefix + 'classification terminee en ' + str(round(time.time()-t0, 1)) + 's')
            pages_typed = []; type_counts = {}
            for p, rep in zip(active_pages, reps1):
                d = parse_json(rep['text'])
                titre = norm_str(d.get('titre'))
                t_model = (norm_str(d.get('type')) or '').upper()
                types, nums = types_from_title(titre)
                if not types:
                    if t_model in ('ACTIF','PASSIF','TCR','DECL'): types = [t_model]
                    else: types = ['AUTRE']
                p['types'] = types; p['annexe_nums'] = nums
                pages_typed.append((p, types))
                label = '+'.join(types); type_counts[label] = type_counts.get(label, 0) + 1
            log(prefix + 'types (contenu): ' + ', '.join(str(k) + '=' + str(v) for k, v in sorted(type_counts.items())))
            del mini, reps1; gc.collect()
            groups = {}
            for p, types in pages_typed: groups.setdefault(tuple(types), []).append(p)
            extracted = {}
            for key_types, plist in groups.items():
                prompt = prompt_cache.get(key_types)
                if prompt is None:
                    prompt = build_prompt(list(key_types)) if key_types != ('AUTRE',) else ('Lis cette page d un dossier fiscal algerien. Extrais en JSON strict: type_page, titre_page, entete, tables (liste de {table_libelle, colonnes, lignes}). ' + chr(10).join(RULES))
                    prompt_cache[key_types] = prompt
                nb_batches = (len(plist) + GPU_BATCH_SIZE - 1) // GPU_BATCH_SIZE
                batch_no = 0
                for batch in chunks(plist, GPU_BATCH_SIZE):
                    batch_no += 1
                    page_nums = [str(p['index'] + 1) for p in batch]
                    log(prefix + 'extraction ' + '+'.join(key_types) + ' batch ' + str(batch_no) + '/' + str(nb_batches) + ' pages [' + ','.join(page_nums) + ']')
                    t0 = time.time()
                    reps = ask_batch(prompt, [p['image'] for p in batch])
                    for p, rep in zip(batch, reps):
                        tok_in += rep['tokens_in']; tok_out += rep['tokens_out']
                        extracted[p['index']] = build_page_object(p, list(key_types), parse_json(rep['text']), rep, pdf_path.name)
                    log(prefix + 'extraction ' + '+'.join(key_types) + ' batch ' + str(batch_no) + ' terminee en ' + str(round(time.time()-t0, 1)) + 's | tokens=' + str(sum(r['tokens_in'] + r['tokens_out'] for r in reps)))
                    gc.collect(); torch.cuda.empty_cache()
            for p in active_pages:
                if p['index'] in extracted: page_objects.append(extracted[p['index']])
                else: page_objects.append(build_page_object(p, ['AUTRE'], {}, None, pdf_path.name))
        page_objects.sort(key=lambda x: x['pdf_page_index'])
        elapsed = time.time() - t_pdf
        result = build_document(pdf_path, pages, page_objects, tok_in, tok_out, elapsed)
        json_path = JSON_DIR / (pdf_path.stem + '.json')
        with open(json_path, 'w', encoding='utf-8') as f: json.dump(result, f, ensure_ascii=False, indent=2, default=str)
        torch.cuda.empty_cache()
        n_ok += 1
        index_records.append({'fichier': pdf_path.name, 'json_path': str(json_path), 'nb_pages': len(pages),
                              'nb_pages_utiles': result['document']['nb_pages_utiles'], 'nb_pages_blanches': result['document']['nb_pages_blanches'],
                              'nb_pages_annexes': result['document']['nb_pages_annexes'], 'tokens_total': tok_in + tok_out,
                              'duree_s': round(elapsed, 2), 'date_extraction': result['document']['date_extraction']})
        eta = (time.time() - t_total) / num * (len(a_traiter) - num)
        log(prefix + 'OK ' + pdf_path.name + ' | ' + str(round(elapsed, 1)) + 's | tok=' + str(tok_in + tok_out) + ' | pages=' + str(len(pages)) + ' | utiles=' + str(result['document']['nb_pages_utiles']) + ' | blanches=' + str(result['document']['nb_pages_blanches']) + ' | annexes=' + str(result['document']['nb_pages_annexes']) + ' | ETA ' + str(round(eta/3600, 2)) + 'h')
    except Exception as e:
        n_err += 1
        import traceback as _tb
        log(prefix + 'ERREUR ' + pdf_path.name + ' — ' + type(e).__name__ + ': ' + str(e))
        with open(LOG_PATH, 'a', encoding='utf-8') as f: f.write(_tb.format_exc() + chr(10))
        continue
with open(INDEX_PATH, 'w', encoding='utf-8') as f:
    for rec in index_records: f.write(json.dumps(rec, ensure_ascii=False) + chr(10))
log('✅ Termine en ' + str(round(time.time() - t_total, 1)) + 's | OK ' + str(n_ok) + ' | Erreurs ' + str(n_err))


In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 11 — VALIDATION RAPIDE | YARISK_V21
# ════════════════════════════════════════════════════════════
files = sorted(JSON_DIR.glob('*.json'))
if not files: print('Aucun JSON pour le moment.')
else:
    d = json.load(open(files[-1], encoding='utf-8'))
    print('Fichier:', d['document']['fichier_source'])
    print('Pages:', d['document']['nb_pages_pdf'], '| utiles:', d['document']['nb_pages_utiles'], '| blanches:', d['document']['nb_pages_blanches'], '| annexes:', d['document']['nb_pages_annexes'])
    for p in d['pages']:
        c = p['classification']
        print(p['page_id'], '| scan', p['numero_page_scannee'], '|', '+'.join(c.get('types') or [c['type_page']]), '| nums', c.get('annexe_numeros'), '| champs:', p['statut_extraction']['nb_champs_extraits'])
    syn = d['synthese']
    print('Synthese: actif', syn['actif'] is not None, '| passif', syn['passif'] is not None, '| tcr', syn['tcr'] is not None, '| decl', syn['decl'] is not None)
    print('Annexes presentes:', ', '.join(sorted(k for k, v in syn['annexes'].items() if v is not None)) or 'aucune')


---
# Partie 2 — Contrôles Ya-Risk et pré-certification Excel V20

Cette partie ne modifie jamais les cellules Excel contenant des formules.
Les données Qwen restent conservées dans le JSON brut ; l’Excel est une vue de pré-certification.


In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 12 — IDENTITE DU DOSSIER | YARISK_V21 
# ════════════════════════════════════════════════════════════
def nif_15(valeur):
    """Valide un NIF sans jamais compléter ni tronquer la lecture Qwen."""
    if valeur is None:
        return None, 'ABSENT'
    chiffres = re.sub(r'\D', '', str(valeur))
    if not chiffres:
        return None, 'ABSENT'
    if len(chiffres) == 15:
        return chiffres, 'CONFORME'
    return None, 'NON_CONFORME_' + str(len(chiffres)) + '_CHIFFRES'
def cle_nom(valeur):
    if not valeur: return None
    s = ''.join(c for c in unicodedata.normalize('NFD', str(valeur)) if unicodedata.category(c) != 'Mn').upper()
    s = re.sub(r'\b(SARL|EURL|SPA|SNC|ETS|STE|SOCIETE|GROUPE)\b', ' ', s)
    s = re.sub(r'[^A-Z0-9]+', ' ', s)
    return ' '.join(s.split()) or None
def annee_exercice(entete):
    for cle in ('exercice', 'exercice_au', 'exercice_du'):
        v = entete.get(cle)
        if not v: continue
        annees = re.findall(r'(19|20)\d{2}', str(v))
        if annees: return int(annees[-1])
    return None
def date_cloture(entete):
    for cle in ('exercice', 'exercice_au'):
        v = entete.get(cle)
        if v and re.search(r'\d{2}/\d{2}/\d{4}', str(v)): return re.search(r'\d{2}/\d{2}/\d{4}', str(v)).group()
    return norm_str(entete.get('exercice')) if entete.get('exercice') else None
def _majoritaire(valeurs):
    vals = [v for v in valeurs if v not in (None, '')]
    if not vals: return None, 0, 0
    compte = Counter(vals); val, n = compte.most_common(1)[0]
    return val, n, len(vals)
def construire_identite(doc):
    par_page, alertes = [], []
    for page in doc.get('pages', []):
        if page['classification'].get('page_blanche'): continue
        ent = page.get('entete_page') or {}
        n15, statut_nif = nif_15(ent.get('nif'))
        par_page.append({'page_id': page['page_id'], 'numero_page_scannee': page['numero_page_scannee'], 'types': page['classification'].get('types') or [],
                         'entreprise': norm_str(ent.get('entreprise')), 'entreprise_cle': cle_nom(ent.get('entreprise')), 'nif_brut': norm_str(ent.get('nif')),
                         'nif_15': n15, 'nif_statut': statut_nif, 'exercice_clos': date_cloture(ent), 'annee_n': annee_exercice(ent)})
    nif_ret, n_nif, t_nif = _majoritaire([p['nif_15'] for p in par_page])
    nom_ret, n_nom, t_nom = _majoritaire([p['entreprise_cle'] for p in par_page])
    an_ret, n_an, t_an = _majoritaire([p['annee_n'] for p in par_page])
    libelle = None
    for p in par_page:
        if p['entreprise_cle'] == nom_ret and p['entreprise']:
            if libelle is None or len(p['entreprise']) > len(libelle): libelle = p['entreprise']
    cloture, _, _ = _majoritaire([p['exercice_clos'] for p in par_page])
    def divergences(champ, retenu):
        return [{'page_id': p['page_id'], 'numero_page_scannee': p['numero_page_scannee'], 'valeur': p[champ]} for p in par_page if p[champ] not in (None, '') and p[champ] != retenu]
    for champ, retenu, libelle_champ, gravite in [('nif_15', nif_ret, 'NIF', 'critique'), ('entreprise_cle', nom_ret, 'raison sociale', 'critique'), ('annee_n', an_ret, 'exercice clos', 'majeur')]:
        div = divergences(champ, retenu)
        if div: alertes.append({'code': 'IDENTITE_DIVERGENTE', 'champ': libelle_champ, 'gravite': gravite, 'valeur_retenue': retenu, 'pages_divergentes': div,
                                'message': str(len(div)) + ' page(s) portent un(e) ' + libelle_champ + ' different(e) de la valeur majoritaire.'})
    couverture = {}
    for champ, libelle_champ in [('nif_15', 'NIF'), ('entreprise', 'raison sociale'), ('annee_n', 'exercice clos')]:
        absentes = [p['page_id'] for p in par_page if not p[champ]]
        couverture[champ] = {'pages_renseignees': len(par_page) - len(absentes), 'pages_totales': len(par_page), 'pages_absentes': absentes}
        if absentes: alertes.append({'code': 'ENTETE_INCOMPLETE', 'champ': libelle_champ, 'gravite': 'mineur', 'pages_sans_valeur': absentes,
                                     'message': str(len(absentes)) + ' page(s) sur ' + str(len(par_page)) + ' sans ' + libelle_champ + ' lisible.'})
    orphelines = [p['page_id'] for p in par_page if not p['nif_15'] and not p['entreprise'] and not p['annee_n']]
    if orphelines: alertes.append({'code': 'PAGE_ORPHELINE', 'champ': 'en-tete', 'gravite': 'majeur', 'pages_sans_valeur': orphelines,
                                   'message': str(len(orphelines)) + ' page(s) sans aucun element d identite.'})
    non_conformes = [{'page_id': p['page_id'], 'statut': p['nif_statut'], 'nif_brut': p['nif_brut']} for p in par_page if str(p['nif_statut']).startswith('NON_CONFORME')]
    if non_conformes: alertes.append({'code': 'NIF_NON_CONFORME', 'champ': 'NIF', 'gravite': 'mineur', 'pages': non_conformes, 'message': 'NIF non conforme : la valeur brute est conservee, aucune completion/troncature automatique.'})
    doc['identite'] = {'entreprise': libelle, 'entreprise_cle': nom_ret, 'nif_15': nif_ret, 'exercice_clos': cloture, 'annee_n': an_ret,
                       'annee_n1': (an_ret - 1) if isinstance(an_ret, int) else None, 'libelle_n': ('N : ' + str(an_ret)) if an_ret else 'N',
                       'libelle_n1': ('N-1 : ' + str(an_ret - 1)) if an_ret else 'N-1',
                       'concordance': {'nif': {'pages_concordantes': n_nif, 'pages_renseignees': t_nif}, 'entreprise': {'pages_concordantes': n_nom, 'pages_renseignees': t_nom}, 'exercice': {'pages_concordantes': n_an, 'pages_renseignees': t_an}},
                       'couverture_entete': couverture, 'dossier_homogene': not any(a['gravite'] == 'critique' for a in alertes), 'pages': par_page, 'alertes': alertes}
    return doc['identite']
print('✅ Identite dossier OK')


In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 13 — REGLES DE CALCUL | YARISK_V21 
# ════════════════════════════════════════════════════════════
COLS_ACTIF = ['montant_brut', 'amortissements_provisions_pertes', 'net_n', 'net_n1']
COLS_PASSIF = ['n', 'n1']
FORMULES = [
 # ── Sous-totaux intermediaires ACTIF ──
 ('ACTIF', 'immobilisations_corporelles', COLS_ACTIF, [('+','terrains'),('+','batiments'),('+','autres_immobilisations_corporelles')]),
 ('ACTIF', 'immobilisations_financieres', COLS_ACTIF, [('+','titres_mis_en_equivalence'),('+','autres_participations_creances'),('+','autres_titres_immobilises'),('+','prets_actifs_financiers_non_courants'),('+','impots_differes_actif')]),
 ('ACTIF', 'creances_emplois_assimiles', COLS_ACTIF, [('+','clients'),('+','autres_debiteurs'),('+','impots_assimiles_actif'),('+','autres_creances_assimiles')]),
 ('ACTIF', 'disponibilites_assimiles', COLS_ACTIF, [('+','placements_financiers_courants'),('+','tresorerie_actif')]),
 # ── Totaux principaux ──
 ('ACTIF', 'total_actif_non_courant', COLS_ACTIF, [('+','ecarts_acquisition_goodwill'),('+','immobilisations_incorporelles'),('+','terrains'),('+','batiments'),('+','autres_immobilisations_corporelles'),('+','immobilisations_en_concession'),('+','immobilisations_en_cours'),('+','titres_mis_en_equivalence'),('+','autres_participations_creances'),('+','autres_titres_immobilises'),('+','prets_actifs_financiers_non_courants'),('+','impots_differes_actif')]),
 ('ACTIF', 'total_actif_courant', COLS_ACTIF, [('+','stocks_encours'),('+','clients'),('+','autres_debiteurs'),('+','impots_assimiles_actif'),('+','autres_creances_assimiles'),('+','placements_financiers_courants'),('+','tresorerie_actif')]),
 ('ACTIF', 'total_general_actif', COLS_ACTIF, [('+','total_actif_non_courant'),('+','total_actif_courant')]),
 ('PASSIF', 'total_capitaux_propres', COLS_PASSIF, [('+','capital_emis'),('-','capital_non_appele'),('+','primes_reserves'),('+','ecart_reevaluation'),('+','ecart_equivalence'),('+','resultat_net_passif'),('+','report_a_nouveau'),('+','part_societe_consolidante'),('+','part_minoritaires')]),
 ('PASSIF', 'total_passifs_non_courants', COLS_PASSIF, [('+','emprunts_dettes_financieres'),('+','impots_differes_provisionnes'),('+','autres_dettes_non_courantes'),('+','provisions_produits_avance')]),
 ('PASSIF', 'total_passifs_courants', COLS_PASSIF, [('+','fournisseurs_rattaches'),('+','impots_passif'),('+','autres_dettes'),('+','tresorerie_passif')]),
 ('PASSIF', 'total_general_passif', COLS_PASSIF, [('+','total_capitaux_propres'),('+','total_passifs_non_courants'),('+','total_passifs_courants')]),
 ('TCR', 'chiffre_affaires_net', 'NET', [('+','ventes_marchandises'),('+','produits_fabriques'),('+','prestations_services'),('+','ventes_travaux'),('+','produits_annexes'),('+','rabais_remises_ristournes_accordes')]),
 ('TCR', 'production_exercice', 'NET', [('+','chiffre_affaires_net'),('+','production_stockee_destockee'),('+','production_immobilisee'),('+','subvention_exploitation')]),
 ('TCR', 'consommations_exercice', 'NET', [('+','achats_marchandises_vendues'),('+','matieres_premieres'),('+','autres_approvisionnements'),('+','variation_stocks'),('+','achats_etudes_prestations'),('+','autres_consommations'),('+','rabais_remises_ristournes_obtenus_achats'),('+','sous_traitance_generale'),('+','locations'),('+','entretien_reparations'),('+','primes_assurances'),('+','personnel_exterieur'),('+','remuneration_intermediaires'),('+','publicite'),('+','deplacements_missions'),('+','rabais_remises_ristournes_obtenus_services'),('+','autres_services')]),
 ('TCR', 'valeur_ajoutee_exploitation', 'NET', [('+','production_exercice'),('+','consommations_exercice')]),
 ('TCR', 'excedent_brut_exploitation', 'NET', [('+','valeur_ajoutee_exploitation'),('+','charges_personnel'),('+','impots_taxes_assimiles')]),
 ('TCR', 'resultat_operationnel', 'NET', [('+','excedent_brut_exploitation'),('+','autres_produits_operationnels'),('+','autres_charges_operationnelles'),('+','dotations_amortissements'),('+','provisions'),('+','pertes_valeur'),('+','reprises_pertes_valeur_provisions')]),
 ('TCR', 'resultat_financier', 'NET', [('+','produits_financiers'),('+','charges_financieres')]),
 ('TCR', 'resultat_ordinaire', 'NET', [('+','resultat_operationnel'),('+','resultat_financier')]),
 ('TCR', 'resultat_extraordinaire', 'NET', [('+','elements_extraordinaires_produits'),('+','elements_extraordinaires_charges')]),
 ('TCR', 'resultat_net_exercice', 'NET', [('+','resultat_ordinaire'),('+','resultat_extraordinaire'),('+','impots_exigibles_resultats'),('+','impots_differes_resultats')]),
 ('A1', 'total', ['solde_debut','debit','credit','solde_fin'], [('+','stocks_marchandises'),('+','matieres_fournitures'),('+','autres_approvisionnements'),('+','encours_production_biens'),('+','encours_production_services'),('+','stocks_produits'),('+','stocks_provenant_immobilisations'),('+','stocks_exterieur')]),
 ('A3', 'total_autres_services', ['montant'], [('+','charges_locatives'),('+','etudes_recherches'),('+','documentation_divers'),('+','transports_biens'),('+','frais_postaux'),('+','services_bancaires'),('+','cotisations_divers')]),
 ('A3', 'total_charges_personnel', ['montant'], [('+','remunerations_personnel'),('+','remuneration_exploitant'),('+','cotisations_sociales'),('+','charges_sociales_exploitant'),('+','autres_charges_sociales'),('+','autres_charges_personnel')]),
 ('A3', 'total_impots', ['montant'], [('+','impots_sur_remunerations'),('+','impots_non_recuperables'),('+','autres_impots_taxes')]),
 ('A3', 'total_general', ['montant'], [('+','total_autres_services'),('+','total_charges_personnel'),('+','total_impots')]),
 ('A4', 'total_charges', ['montant'], [('+','redevances_concessions_charges'),('+','moins_values_sorties_actifs'),('+','jetons_presence_charges'),('+','pertes_creances_irrecouvrables'),('+','quote_part_operations_commun_charges'),('+','amendes_penalites_dons'),('+','charges_exceptionnelles_gestion'),('+','autres_charges_gestion')]),
 ('A4', 'total_produits', ['montant'], [('+','redevances_concessions_produits'),('+','plus_values_sorties_actifs'),('+','jetons_presence_produits'),('+','quotes_parts_subventions_virees'),('+','quote_part_operations_commun_produits'),('+','rentrees_creances_amorties'),('+','produits_exceptionnels_gestion'),('+','autres_produits_gestion')]),
 ('A5', 'total', ['dotations_cumulees_debut','dotations_exercice','diminutions_elements_sortis','dotations_cumulees_fin','dotations_fiscales_exercice','ecarts'], [('+','goodwill'),('+','immobilisations_incorporelles'),('+','immobilisations_corporelles'),('+','participations'),('+','autres_actifs_financiers_non_courants')]),
 ('A6', 'total', ['montants_bruts','tva_deduite','montant_net_a_amortir'], [('+','goodwill'),('+','immobilisations_incorporelles'),('+','immobilisations_corporelles'),('+','participations'),('+','autres_actifs_financiers_non_courants')]),
 ('A8', 'total', ['provisions_cumulees_debut','dotations_exercice','reprises_exercice','provisions_cumulees_fin'], [('+','pertes_valeur_stocks'),('+','pertes_valeur_creances'),('+','pertes_valeur_actions'),('+','provisions_pensions'),('+','provisions_litiges'),('+','autres_provisions_personnel'),('+','provisions_impots'),('+','autres_provisions')]),
 ('A9', 'total_reintegrations', ['montant'], [('+','charges_immeubles_non_affectes'),('+','quote_part_cadeaux_publicitaires'),('+','quote_part_sponsoring'),('+','frais_reception'),('+','cotisations_dons'),('+','impots_taxes_non_deductibles'),('+','provisions_non_deductibles'),('+','amortissements_non_deductibles'),('+','quote_part_frais_rd'),('+','amortissements_credit_bail_preneur'),('+','loyers_hors_produits_financiers_bailleur'),('+','ibs_impot_exigible'),('+','ibs_impot_differe'),('+','pertes_valeur_non_deductibles'),('+','amendes_penalites'),('+','autres_reintegrations')]),
 ('A9', 'total_deductions', ['montant'], [('+','plus_values_cession_actif_immobilise'),('+','produits_plus_values_actions_bourse'),('+','revenus_distribution_benefices'),('+','amortissements_credit_bail_bailleur'),('+','loyers_hors_charges_financieres_preneur'),('+','complement_amortissements'),('+','autres_deductions')]),
 ('A10', 'origine_total', ['montant'], [('+','origine_report_a_nouveau_n1'),('+','origine_resultat_n1'),('+','origine_prelevements_reserves')]),
 ('A10', 'affectation_total', ['montant'], [('+','affectation_reserves'),('+','affectation_augmentation_capital'),('+','affectation_dividendes'),('+','affectation_report_a_nouveau')]),
]
FORMULES_LIGNE = [
 ('ACTIF', 'net_n', [('+','montant_brut'),('-','amortissements_provisions_pertes')]),
 ('A1', 'solde_fin', [('+','solde_debut'),('+','debit'),('-','credit')]),
 ('A5', 'dotations_cumulees_fin', [('+','dotations_cumulees_debut'),('+','dotations_exercice'),('-','diminutions_elements_sortis')]),
 ('A5', 'ecarts', [('+','dotations_exercice'),('-','dotations_fiscales_exercice')]),
 ('A7', 'valeur_nette_comptable', [('+','montant_net_actif'),('-','amortissements_pratiques')]),
 ('A8', 'provisions_cumulees_fin', [('+','provisions_cumulees_debut'),('+','dotations_exercice'),('-','reprises_exercice')]),
 ('A82', 'valeur_nette_comptable', [('+','valeur_nominale_debut'),('-','perte_valeur_constituee')]),
]
CONTROLES = [
 ('Equilibre du bilan (N)', 'critique', [('+','ACTIF','total_general_actif','net_n')], [('+','PASSIF','total_general_passif','n')]),
 ('Equilibre du bilan (N-1)', 'critique', [('+','ACTIF','total_general_actif','net_n1')], [('+','PASSIF','total_general_passif','n1')]),
 ('Resultat net : TCR = bilan passif (N)', 'critique', [('+','TCR','resultat_net_exercice','NET')], [('+','PASSIF','resultat_net_passif','n')]),
 ('Resultat net : TCR = bilan passif (N-1)', 'critique', [('+','TCR','resultat_net_exercice','NET_N1')], [('+','PASSIF','resultat_net_passif','n1')]),
 ('Resultat net : TCR = ligne I du tableau 9', 'critique', [('+','TCR','resultat_net_exercice','NET')], [('+','A9','resultat_net_benefice','montant'),('-','A9','resultat_net_perte','montant')]),
 ('Stocks : total A1 (fin) = stocks bilan (N)', 'majeur', [('+','A1','total','solde_fin')], [('+','ACTIF','stocks_encours','net_n')]),
 ('Stocks : total A1 (debut) = stocks bilan (N-1)', 'majeur', [('+','A1','total','solde_debut')], [('+','ACTIF','stocks_encours','net_n1')]),
 ('Production stockee : solde A2 = ligne TCR', 'majeur', [('+','A2','production_stockee','solde_debiteur'),('-','A2','production_stockee','solde_crediteur')], [('+','TCR','production_stockee_destockee','n_debit'),('-','TCR','production_stockee_destockee','n_credit')]),
 ('Amortissements : cumul fin A5 = colonne amort. bilan', 'majeur', [('+','A5','total','dotations_cumulees_fin')], [('+','ACTIF','total_actif_non_courant','amortissements_provisions_pertes')]),
 ('Amortissements : dotations A5 = dotations TCR', 'majeur', [('+','A5','total','dotations_exercice')], [('+','TCR','dotations_amortissements','n_debit')]),
 ('Charges A3 : TOTAL(1) = autres services TCR', 'majeur', [('+','A3','total_autres_services','montant')], [('+','TCR','autres_services','n_debit')]),
 ('Charges A3 : TOTAL(2) = charges personnel TCR', 'majeur', [('+','A3','total_charges_personnel','montant')], [('+','TCR','charges_personnel','n_debit')]),
 ('Charges A3 : TOTAL(3) = impots et taxes TCR', 'majeur', [('+','A3','total_impots','montant')], [('+','TCR','impots_taxes_assimiles','n_debit')]),
 ('A4 : total charges = autres charges TCR', 'majeur', [('+','A4','total_charges','montant')], [('+','TCR','autres_charges_operationnelles','n_debit')]),
 ('A4 : total produits = autres produits TCR', 'majeur', [('+','A4','total_produits','montant')], [('+','TCR','autres_produits_operationnels','n_credit')]),
 ('Cessions : plus-values A7 = tableau 4', 'majeur', [('+','A7',' TOTAL ','plus_value')], [('+','A4','plus_values_sorties_actifs','montant')]),
 ('Cessions : moins-values A7 = tableau 4', 'majeur', [('+','A7',' TOTAL ','moins_value')], [('+','A4','moins_values_sorties_actifs','montant')]),
 ('Provisions : dotations A8 = provisions+pertes TCR', 'majeur', [('+','A8','total','dotations_exercice')], [('+','TCR','provisions','n_debit'),('+','TCR','pertes_valeur','n_debit')]),
 ('Provisions : reprises A8 = reprises TCR', 'majeur', [('+','A8','total','reprises_exercice')], [('+','TCR','reprises_pertes_valeur_provisions','n_credit')]),
 ('Provisions : 8/1 = pertes creances tableau 8', 'majeur', [('+','A81',' TOTAL ','perte_valeur_constituee')], [('+','A8','pertes_valeur_creances','provisions_cumulees_fin')]),
 ('Provisions : 8/2 = pertes actions tableau 8', 'majeur', [('+','A82',' TOTAL ','perte_valeur_constituee')], [('+','A8','pertes_valeur_actions','provisions_cumulees_fin')]),
 ('Fiscal : IBS exigible = impots exigibles TCR', 'majeur', [('+','A9','ibs_impot_exigible','montant')], [('+','TCR','impots_exigibles_resultats','n_debit')]),
 ('Fiscal : resultat = I + reint - ded - deficits', 'critique', [('+','A9','resultat_fiscal_benefice','montant'),('-','A9','resultat_fiscal_deficit','montant')], [('+','A9','resultat_net_benefice','montant'),('-','A9','resultat_net_perte','montant'),('+','A9','total_reintegrations','montant'),('-','A9','total_deductions','montant'),('-','A9','total_deficits_a_deduire','montant')]),
 ('Fiscal : resultat fiscal DECL = tableau 9', 'majeur', [('+','DECL','resultat_fiscal','valeur')], [('+','A9','resultat_fiscal_benefice','montant'),('-','A9','resultat_fiscal_deficit','montant')]),
 ('Fiscal : resultat comptable DECL = RN TCR', 'majeur', [('+','DECL','resultat_comptable','valeur')], [('+','TCR','resultat_net_exercice','NET')]),
 ('Fiscal : CA global DECL = CA net TCR', 'majeur', [('+','DECL','chiffre_affaires_global_ht','valeur')], [('+','TCR','chiffre_affaires_net','NET')]),
 ('Affectation : origine = affectation', 'critique', [('+','A10','origine_total','montant')], [('+','A10','affectation_total','montant')]),
 ('Affectation : resultat N-1 = RN bilan (N-1)', 'majeur', [('+','A10','origine_resultat_n1','montant')], [('+','PASSIF','resultat_net_passif','n1')]),
 ('Affectation : reserves = variation primes/reserves', 'majeur', [('+','A10','affectation_reserves','montant')], [('+','PASSIF','primes_reserves','n'),('-','PASSIF','primes_reserves','n1')]),
 ('Affectation : augm. capital = variation capital', 'majeur', [('+','A10','affectation_augmentation_capital','montant')], [('+','PASSIF','capital_emis','n'),('-','PASSIF','capital_emis','n1')]),
 ('Tableau 12 = remuneration intermediaires TCR', 'majeur', [('+','A12',' TOTAL ','montant_percu')], [('+','TCR','remuneration_intermediaires','n_debit')]),
 ('Distributions : global = societe + etablissement', 'majeur', [('+','DIST','montant_global_brut','montant')], [('+','DIST','paye_par_societe','montant'),('+','DIST','paye_par_etablissement','montant')]),
 ("Amortissements : ecart A5 = reintegration au tableau 9", 'indicatif',
  [('+','A5','total','ecarts')], [('+','A9','amortissements_non_deductibles','montant')]),
 ("Cessions : plus-values A7 = deduction au tableau 9", 'indicatif',
  [('+','A7','__TOTAL__','plus_value')], [('+','A9','plus_values_cession_actif_immobilise','montant')]),
 ("Cessions : sorties A7 = diminutions du tableau 5", 'indicatif',
  [('+','A7','__TOTAL__','amortissements_pratiques')], [('+','A5','total','diminutions_elements_sortis')]),
 ("Fiscal : IBS differe reintegre = impots differes du TCR", 'majeur',
  [('+','A9','ibs_impot_differe','montant')], [('+','TCR','impots_differes_resultats','n_debit')]),
 ("Affectation : report a nouveau = report du bilan (N)", 'indicatif',
  [('+','A10','affectation_report_a_nouveau','montant')], [('+','PASSIF','report_a_nouveau','n')]),
 ("Affectation : dividendes = distributions du tableau DIST", 'indicatif',
  [('+','A10','affectation_dividendes','montant')], [('+','DIST','montant_global_brut','montant')]),
 ("Sous-traitance : tableau ST = sous-traitance generale du TCR", 'indicatif',
  [('+','ST','__TOTAL__','montant')], [('+','TCR','sous_traitance_generale','n_debit')]),
 ("TAP : CA impose + exonere = chiffre d affaires net du TCR", 'indicatif',
  [('+','A13','__TOTAL__','ca_imposable'),('+','A13','__TOTAL__','ca_exonere')],
  [('+','TCR','chiffre_affaires_net','NET')]),
 ("Participations : valeur des titres = poste du bilan", 'indicatif',
  [('+','A11','__TOTAL__','valeur_comptable_titres')], [('+','ACTIF','autres_participations_creances','net_n')]),
]
print('✅ Regles OK — ' + str(len(FORMULES)) + ' formules, ' + str(len(FORMULES_LIGNE)) + ' formules ligne, ' + str(len(CONTROLES)) + ' controles')


In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 14 — MOTEUR DE COHERENCE | YARISK_V21
# - null ≠ 0
# - contrôle non vérifiable si un composant requis manque
# - garde-fous TCR Débit/Crédit
# ════════════════════════════════════════════════════════════
TOLERANCE_DA = 1000.0   # seuil metier : les ecarts <= 1000 DZD ne sont pas signales

def lire_bloc(bloc):
    if isinstance(bloc, list):
        sortie = []
        for i, row in enumerate(bloc):
            if not isinstance(row, dict):
                continue
            vals = {k: v for k, v in row.items() if k != 'libelle_imprime'}
            sortie.append(('ligne_' + str(i).zfill(3), vals, row.get('libelle_imprime')))
        return sortie
    if isinstance(bloc, dict):
        if isinstance(bloc.get('lignes'), list):
            sortie = []
            for i, lg in enumerate(bloc['lignes']):
                if not isinstance(lg, dict):
                    continue
                cle = lg.get('row_code') or ('ligne_' + str(i).zfill(3))
                sortie.append((cle, lg.get('valeurs') or {}, lg.get('libelle_imprime')))
            return sortie
        return [(rc, vals, None) for rc, vals in bloc.items() if isinstance(vals, dict)]
    return []

def iter_blocs(doc):
    for page in doc.get('pages', []):
        for tab, bloc in (page.get('donnees') or {}).items():
            if tab in ('brut', 'AUTRE') or bloc in (None, {}, []):
                continue
            yield page, tab, bloc

def _index_postes(doc):
    idx, dyn = {}, {}
    for _page, tab, bloc in iter_blocs(doc):
        for cle, vals, _lib in lire_bloc(bloc):
            renseignees = {k: v for k, v in (vals or {}).items() if v is not None}
            if renseignees:
                idx.setdefault((tab, cle), {}).update(renseignees)
            if cle != 'total':
                for col, v in renseignees.items():
                    if isinstance(v, (int, float)) and not isinstance(v, bool):
                        dyn[(tab, col)] = dyn.get((tab, col), 0.0) + float(v)
    for (tab, col), somme in dyn.items():
        idx.setdefault((tab, ' TOTAL '), {})[col] = somme
    return idx

def _valeur(idx, tab, rc, col):
    vals = idx.get((tab, rc))
    if vals is None:
        return None
    if col in ('NET', 'NET_N1'):
        suffixe = ('n_credit', 'n_debit') if col == 'NET' else ('n1_credit', 'n1_debit')
        c, d = vals.get(suffixe[0]), vals.get(suffixe[1])
        if c is None and d is None:
            return None
        # Ici, null n'est pas remplacé par zéro pour un contrôle de complétude.
        # Le NET est calculable si au moins un côté est renseigné : sur le TCR
        # l'autre côté est structurellement vide pour un solde donné.
        c = float(c) if isinstance(c, (int, float)) and not isinstance(c, bool) else 0.0
        d = float(d) if isinstance(d, (int, float)) and not isinstance(d, bool) else 0.0
        return c - d
    v = vals.get(col)
    if isinstance(v, bool) or not isinstance(v, (int, float)):
        return None
    return float(v)

def _somme_detail(idx, termes, exiger_complet=True):
    total = 0.0
    vus = 0
    manquants = []
    for signe, tab, rc, col in termes:
        v = _valeur(idx, tab, rc, col)
        if v is None:
            manquants.append(tab + '/' + rc + '/' + col)
            continue
        total += v if signe == '+' else -v
        vus += 1
    if vus == 0:
        return None, manquants
    if exiger_complet and manquants:
        return None, manquants
    return total, manquants

def _somme(idx, termes, exiger_complet=True):
    return _somme_detail(idx, termes, exiger_complet=exiger_complet)[0]

def _statut(ecart, tol):
    if abs(ecart) <= 1e-9:
        return 'coherent'
    if abs(ecart) <= tol:
        return 'coherent_arrondi'
    return 'ecart_significatif'

def verifier_coherence(doc, tolerance=TOLERANCE_DA):
    idx = _index_postes(doc)
    resultats_formules, resultats_controles = [], []

    # ── Formules agrégées : comparaison uniquement si TOUS les composants requis sont lus
    for tab, cible, cols, comps in FORMULES:
        colonnes = ['NET', 'NET_N1'] if cols == 'NET' else cols
        for col in colonnes:
            declaree = _valeur(idx, tab, cible, col)
            if declaree is None:
                continue
            recalc, manquants = _somme_detail(
                idx, [(s, tab, rc, col) for s, rc in comps], exiger_complet=True
            )
            if recalc is None:
                resultats_formules.append({
                    'type': 'agregat', 'tableau': tab, 'poste': cible, 'colonne': col,
                    'valeur_extraite': declaree, 'valeur_recalculee': None, 'ecart': None,
                    'statut': 'non_verifiable', 'motif': 'COMPOSANTS_MANQUANTS',
                    'postes_manquants': manquants
                })
                continue
            ecart = declaree - recalc
            resultats_formules.append({
                'type': 'agregat', 'tableau': tab, 'poste': cible, 'colonne': col,
                'valeur_extraite': declaree, 'valeur_recalculee': recalc,
                'ecart': round(ecart, 2), 'statut': _statut(ecart, tolerance)
            })

    # ── Formules ligne-à-ligne
    for tab, col_cible, comps in FORMULES_LIGNE:
        for (t, rc) in list(idx.keys()):
            if t != tab or rc == ' TOTAL ':
                continue
            declaree = _valeur(idx, tab, rc, col_cible)
            if declaree is None:
                continue
            recalc, manquants = _somme_detail(
                idx, [(s, tab, rc, c) for s, c in comps], exiger_complet=True
            )
            if recalc is None:
                resultats_formules.append({
                    'type': 'ligne', 'tableau': tab, 'poste': rc, 'colonne': col_cible,
                    'valeur_extraite': declaree, 'valeur_recalculee': None, 'ecart': None,
                    'statut': 'non_verifiable', 'motif': 'COMPOSANTS_MANQUANTS',
                    'postes_manquants': manquants
                })
                continue
            ecart = declaree - recalc
            resultats_formules.append({
                'type': 'ligne', 'tableau': tab, 'poste': rc, 'colonne': col_cible,
                'valeur_extraite': declaree, 'valeur_recalculee': recalc,
                'ecart': round(ecart, 2), 'statut': _statut(ecart, tolerance)
            })

    # ── Contrôles croisés
    for libelle, gravite, gauche, droite in CONTROLES:
        a, manq_a = _somme_detail(idx, gauche, exiger_complet=True)
        b, manq_b = _somme_detail(idx, droite, exiger_complet=True)
        if a is None or b is None:
            resultats_controles.append({
                'controle': libelle, 'gravite': gravite, 'statut': 'non_verifiable',
                'valeur_a': a, 'valeur_b': b, 'ecart': None,
                'motif': 'COMPOSANTS_MANQUANTS',
                'postes_manquants': sorted(set(manq_a + manq_b)),
                'commentaire': 'Controle incomplet : au moins un composant requis est absent de l extraction.'
            })
            continue
        ecart = a - b
        resultats_controles.append({
            'controle': libelle, 'gravite': gravite,
            'valeur_a': round(a, 2), 'valeur_b': round(b, 2),
            'ecart': round(ecart, 2), 'statut': _statut(ecart, tolerance)
        })

    ec_f = [f for f in resultats_formules if f['statut'] == 'ecart_significatif']
    nv_f = [f for f in resultats_formules if f['statut'] == 'non_verifiable']
    ec_c = [c for c in resultats_controles if c['statut'] == 'ecart_significatif']
    nv_c = [c for c in resultats_controles if c['statut'] == 'non_verifiable']
    critiques = [c for c in ec_c if c['gravite'] == 'critique']
    critiques_nv = [c for c in nv_c if c['gravite'] == 'critique']

    return {
        'tolerance_da': tolerance,
        'formules': resultats_formules,
        'controles_croises': resultats_controles,
        'synthese': {
            'formules_verifiees': len([f for f in resultats_formules if f['statut'] != 'non_verifiable']),
            'formules_non_verifiables': len(nv_f),
            'formules_en_ecart': len(ec_f),
            'controles_verifies': len([c for c in resultats_controles if c['statut'] != 'non_verifiable']),
            'controles_non_verifiables': len(nv_c),
            'controles_en_ecart': len(ec_c),
            'ecarts_critiques': len(critiques),
            'controles_critiques_non_verifiables': len(critiques_nv),
            'arbitrage_requis': bool(ec_f or ec_c or critiques_nv),
            'liasse_exploitable': not critiques
        }
    }

def analyser_lecture_tcr(doc):
    """Contrôles de lecture TCR sans déplacer ni corriger les valeurs Qwen."""
    idx = _index_postes(doc)
    suspects = []
    totaux_non_lus = []

    for (tab, rc), vals in idx.items():
        if tab != 'TCR' or rc == ' TOTAL ' or rc not in TCR_SENS:
            continue
        sens = TCR_SENS.get(rc, 'B')
        if sens == 'B':
            continue
        for exercice, c_debit, c_credit in (
            ('N', 'n_debit', 'n_credit'),
            ('N-1', 'n1_debit', 'n1_credit')
        ):
            d = vals.get(c_debit)
            c = vals.get(c_credit)
            d_num = isinstance(d, (int, float)) and not isinstance(d, bool)
            c_num = isinstance(c, (int, float)) and not isinstance(c, bool)

            if sens == 'C' and d_num and not c_num:
                suspects.append({
                    'poste': rc, 'exercice': exercice, 'valeur_lue': float(d),
                    'lu_en': 'DEBIT', 'sens_habituel': 'CREDIT',
                    'statut': 'SENS_INHABITUEL_A_VERIFIER'
                })
            elif sens == 'D' and c_num and not d_num:
                suspects.append({
                    'poste': rc, 'exercice': exercice, 'valeur_lue': float(c),
                    'lu_en': 'CREDIT', 'sens_habituel': 'DEBIT',
                    'statut': 'SENS_INHABITUEL_A_VERIFIER'
                })

    for rc in sorted(TCR_TOTAUX_ATTENDUS):
        vals = idx.get(('TCR', rc), {})
        for exercice, c_debit, c_credit in (
            ('N', 'n_debit', 'n_credit'),
            ('N-1', 'n1_debit', 'n1_credit')
        ):
            d, c = vals.get(c_debit), vals.get(c_credit)
            if not (isinstance(d, (int, float)) and not isinstance(d, bool)) and \
               not (isinstance(c, (int, float)) and not isinstance(c, bool)):
                totaux_non_lus.append({
                    'poste': rc, 'exercice': exercice,
                    'statut': 'TOTAL_NON_LU',
                    'commentaire': 'Ligne de total/solde attendue mais aucune valeur Debut/Credit n a ete extraite.'
                })

    return {
        'suspects_sens_debit_credit': suspects,
        'totaux_non_lus': totaux_non_lus,
        'nb_suspects': len(suspects),
        'nb_totaux_non_lus': len(totaux_non_lus)
    }

def enrichir_lignes(doc, rapport):
    index = {}
    for f in rapport['formules']:
        index[(f['tableau'], f['poste'], f['colonne'])] = f

    for page, tab, bloc in iter_blocs(doc):
        cible = page.setdefault('controles_donnees', {}).setdefault(tab, {})
        for cle, vals, _lib in lire_bloc(bloc):
            ctrl = {}
            for col, val in (vals or {}).items():
                if val is None or not isinstance(val, (int, float)) or isinstance(val, bool):
                    continue
                f = index.get((tab, cle, col))
                if f:
                    ctrl[col] = {
                        'valeur_certaine': True if f['statut'] in ('coherent', 'coherent_arrondi') else (False if f['statut'] == 'ecart_significatif' else None),
                        'valeur_extraite': f.get('valeur_extraite'),
                        'valeur_recalculee': f.get('valeur_recalculee'),
                        'ecart': f.get('ecart'),
                        'statut': f['statut'],
                        'motif': f.get('motif'),
                        'postes_manquants': f.get('postes_manquants')
                    }
                else:
                    ctrl[col] = {'valeur_certaine': None, 'statut': 'non_verifiable'}
            if ctrl:
                cible[cle] = ctrl
        if not cible:
            page['controles_donnees'].pop(tab, None)
    return doc

def appliquer_controles(doc):
    identite = construire_identite(doc)
    rapport = verifier_coherence(doc)
    rapport['lecture_tcr'] = analyser_lecture_tcr(doc)

    enrichir_lignes(doc, rapport)
    doc['controles'] = rapport
    doc['schema_version'] = '21.0'
    doc['workflow'] = {'statut': 'A_CERTIFIER', 'certifie': False}

    s = rapport['synthese']
    lecture = rapport['lecture_tcr']
    s['identite_homogene'] = identite['dossier_homogene']
    s['arbitrage_requis'] = bool(
        s['arbitrage_requis']
        or not identite['dossier_homogene']
        or lecture['nb_suspects']
        or lecture['nb_totaux_non_lus']
    )
    return doc

print('✅ Moteur de coherence Ya-Risk V21 OK')


In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 15 — MAPPING EXCEL ANALYSTE V20 | YARISK_V21
# Source de mapping : OfficeScript_MultiExercices_v23
# ════════════════════════════════════════════════════════════

CFG_V20 = {'ACTIF_GROUPES': {'N': ['P', 'Q', 'R'], 'N-1': ['M', 'N', 'O'], 'N-2': ['J', 'K', 'L'], 'N-3': ['G', 'H', 'I'], 'N-4': ['D', 'E', 'F']},
 'PASSIF_COLS': {'N': 'H', 'N-1': 'G', 'N-2': 'F', 'N-3': 'E', 'N-4': 'D'},
 'TCR_PAIRES': {'N': ['M', 'N'], 'N-1': ['K', 'L'], 'N-2': ['I', 'J'], 'N-3': ['G', 'H'], 'N-4': ['E', 'F']},
 'ACTIF_ROWS': {'ecarts_acquisition_goodwill': 10,
                'immobilisations_incorporelles': 11,
                'immobilisations_corporelles': 12,
                'terrains': 13,
                'batiments': 14,
                'autres_immobilisations_corporelles': 15,
                'immobilisations_en_concession': 16,
                'immobilisations_en_cours': 17,
                'immobilisations_financieres': 18,
                'titres_mis_en_equivalence': 19,
                'autres_participations_creances': 20,
                'autres_titres_immobilises': 21,
                'prets_actifs_financiers_non_courants': 22,
                'impots_differes_actif': 23,
                'total_actif_non_courant': 24,
                'stocks_encours': 26,
                'creances_emplois_assimiles': 27,
                'clients': 28,
                'autres_debiteurs': 29,
                'impots_assimiles_actif': 30,
                'autres_creances_assimiles': 31,
                'disponibilites_assimiles': 32,
                'placements_financiers_courants': 33,
                'tresorerie_actif': 34,
                'total_actif_courant': 35,
                'total_general_actif': 36},
 'PASSIF_ROWS': {'capital_emis': 6,
                 'capital_non_appele': 7,
                 'primes_reserves': 8,
                 'ecart_reevaluation': 9,
                 'ecart_equivalence': 10,
                 'resultat_net_passif': 11,
                 'report_a_nouveau': 12,
                 'part_societe_consolidante': 13,
                 'part_minoritaires': 14,
                 'total_capitaux_propres': 15,
                 'emprunts_dettes_financieres': 17,
                 'impots_differes_provisionnes': 18,
                 'autres_dettes_non_courantes': 19,
                 'provisions_produits_avance': 20,
                 'total_passifs_non_courants': 21,
                 'fournisseurs_rattaches': 23,
                 'impots_passif': 24,
                 'autres_dettes': 25,
                 'tresorerie_passif': 26,
                 'total_passifs_courants': 27,
                 'total_general_passif': 28},
 'TCR_ROWS': {'ventes_marchandises': 6,
              'produits_fabriques': 7,
              'prestations_services': 8,
              'ventes_travaux': 9,
              'produits_annexes': 10,
              'rabais_remises_ristournes_accordes': 11,
              'chiffre_affaires_net': 12,
              'production_stockee_destockee': 13,
              'production_immobilisee': 14,
              'subvention_exploitation': 15,
              'production_exercice': 16,
              'achats_marchandises_vendues': 17,
              'matieres_premieres': 18,
              'autres_approvisionnements': 19,
              'variation_stocks': 20,
              'achats_etudes_prestations': 21,
              'autres_consommations': 22,
              'rabais_remises_ristournes_obtenus_achats': 23,
              'sous_traitance_generale': 24,
              'locations': 25,
              'entretien_reparations': 26,
              'primes_assurances': 27,
              'personnel_exterieur': 28,
              'remuneration_intermediaires': 29,
              'publicite': 30,
              'deplacements_missions': 31,
              'autres_services': 32,
              'rabais_remises_ristournes_obtenus_services': 33,
              'consommations_exercice': 34,
              'valeur_ajoutee_exploitation': 35,
              'charges_personnel': 36,
              'impots_taxes_assimiles': 37,
              'excedent_brut_exploitation': 38,
              'autres_produits_operationnels': 39,
              'autres_charges_operationnelles': 40,
              'dotations_amortissements': 41,
              'provisions': 42,
              'pertes_valeur': 43,
              'reprises_pertes_valeur_provisions': 44,
              'resultat_operationnel': 45,
              'produits_financiers': 46,
              'charges_financieres': 47,
              'resultat_financier': 48,
              'resultat_ordinaire': 49,
              'elements_extraordinaires_produits': 50,
              'elements_extraordinaires_charges': 51,
              'resultat_extraordinaire': 52,
              'impots_exigibles_resultats': 53,
              'impots_differes_resultats': 54,
              'resultat_net_exercice': 55},
 'ACTIF_FORMULES': [9, 12, 18, 24, 25, 27, 32, 35, 36],
 'PASSIF_FORMULES': [5, 15, 16, 21, 22, 27, 28],
 'TCR_FORMULES': [12, 16, 34, 35, 38, 45, 48, 49, 52, 55]}

FEUILLE_ACTIF_V20 = 'Saisie actif'
FEUILLE_PASSIF_V20 = 'Saisie passif'
FEUILLE_TCR_V20 = 'Saisie TCR'
FEUILLE_EXTRAIT = '0.Données extraites'
FEUILLE_CHECK = 'Checklist'

# V21 mono-PDF : on alimente le rang N dans le modèle analyste.
# Les valeurs N-1 restent visibles dans 0.Données extraites.
# La phase multi-PDF utilisera les groupes N-1 ... N-4 avec leurs propres brut/amortissements.
RANG_PRECERT = 'N'

TOLERANCE_EXCEL_KDZD = 1.0    # 1 KDZD = 1000 DZD
SEUIL_ECART_IMPORTANT_KDZD = 2.0

def _kdzd(v):
    if isinstance(v, bool) or not isinstance(v, (int, float)):
        return None
    return float(v) / 1000.0

def _source_tableau(doc, code):
    pages = []
    fichiers = []
    for p in doc.get('pages', []):
        if code in (p.get('classification') or {}).get('types', []):
            pages.append(p.get('numero_page_scannee'))
            fs = p.get('fichier_source') or (doc.get('document') or {}).get('fichier_source')
            if fs and fs not in fichiers:
                fichiers.append(fs)
    return {
        'fichiers': fichiers,
        'pages': [x for x in pages if x is not None],
        'texte': (' + '.join(fichiers) if fichiers else '?') +
                 ((' — p.' + ','.join(str(x) for x in pages if x is not None)) if pages else '')
    }

print('✅ Mapping Excel Analyste V20 charge')


In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 16 — PRE-CERTIFICATION EXCEL V20 | YARISK_V21
# - modèle analyste V20
# - aucune formule écrasée
# - données Qwen en DZD -> saisie V20 en KDZD
# - commentaires d'écart sur cellules calculées
# - 0.Données extraites + Checklist
# ════════════════════════════════════════════════════════════

MODEL_DIR = Path('/mnt/Risk/Model')

def _resoudre_modele_v20():
    explicites = [
        MODEL_DIR / 'MODEL ETAT FINANCIER - ANALYSTE COPILOTE V20.xlsx',
        MODEL_DIR / 'MODEL ETAT FINANCIER - ANALYSTE V20.xlsx',
    ]
    for p in explicites:
        if p.exists():
            return p
    candidats = sorted(MODEL_DIR.glob('*ANALYSTE*V20*.xlsx'))
    return candidats[0] if candidats else explicites[0]

MODELE_XLSX = _resoudre_modele_v20()
RECALC_TOOL = MODEL_DIR / 'recalc_liasse.py'
RECALC_DISPONIBLE = RECALC_TOOL.exists() and bool(shutil.which('soffice'))

_ROUGE = 'FF0000'
_ORANGE = 'ED7D31'
_JAUNE = 'FFF2CC'
_ROUGE_FOND = PatternFill('solid', start_color='FFC7CE')
_ORANGE_FOND = PatternFill('solid', start_color='F8CBAD')
_JAUNE_FOND = PatternFill('solid', start_color='FFF2CC')
_VERT_FOND = PatternFill('solid', start_color='C6EFCE')

def _est_formule(cell):
    return isinstance(cell.value, str) and cell.value.startswith('=')

def _bordure_couleur(cell, couleur, style='medium'):
    side = Side(style=style, color=couleur)
    cell.border = Border(left=side, right=side, top=side, bottom=side)

def _scan_formules_invalides(wb):
    problemes = []
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for c in row:
                if isinstance(c.value, str) and c.value.startswith('=') and '#REF!' in c.value.upper():
                    problemes.append(ws.title + '!' + c.coordinate + ' = ' + c.value)
    return problemes

def _recalculer_excel(path_xlsx, timeout=180):
    if not RECALC_DISPONIBLE:
        return {'ok': False, 'code': None, 'message': 'LibreOffice/recalc_liasse.py indisponible'}
    try:
        r = subprocess.run(
            [sys.executable, str(RECALC_TOOL), str(path_xlsx), str(timeout)],
            capture_output=True, text=True, timeout=timeout + 120
        )
        return {
            'ok': r.returncode == 0,
            'code': r.returncode,
            'stdout': (r.stdout or '')[-2000:],
            'stderr': (r.stderr or '')[-2000:],
            'message': 'OK' if r.returncode == 0 else 'recalc_liasse.py code ' + str(r.returncode)
        }
    except Exception as e:
        return {'ok': False, 'code': None, 'message': type(e).__name__ + ': ' + str(e)}

def _vider_cellule_saisie(cell):
    if not _est_formule(cell):
        cell.value = None
        cell.comment = None

def _preparer_zone_n(wb):
    """Réinitialise uniquement le rang N, jamais les cellules formule."""
    wsA, wsP, wsT = wb[FEUILLE_ACTIF_V20], wb[FEUILLE_PASSIF_V20], wb[FEUILLE_TCR_V20]
    gA = CFG_V20['ACTIF_GROUPES'][RANG_PRECERT]
    cP = CFG_V20['PASSIF_COLS'][RANG_PRECERT]
    gT = CFG_V20['TCR_PAIRES'][RANG_PRECERT]

    for rc, rw in CFG_V20['ACTIF_ROWS'].items():
        for col in gA:
            _vider_cellule_saisie(wsA[col + str(rw)])
    for rc, rw in CFG_V20['PASSIF_ROWS'].items():
        _vider_cellule_saisie(wsP[cP + str(rw)])
    for rc, rw in CFG_V20['TCR_ROWS'].items():
        for col in gT:
            _vider_cellule_saisie(wsT[col + str(rw)])

    for addr in (gA[0]+'1', gA[0]+'2', gA[0]+'3', gA[0]+'7',
                 cP+'1', cP+'2', cP+'3',
                 gT[0]+'1', gT[0]+'2', gT[0]+'3'):
        ws = wsA if addr[0] in gA else (wsP if addr[0] == cP else wsT)
        _vider_cellule_saisie(ws[addr])

def _ecrire_identification_v20(wb, doc):
    ident = doc.get('identite') or {}
    sourceA = _source_tableau(doc, 'ACTIF')['texte']
    sourceP = _source_tableau(doc, 'PASSIF')['texte']
    sourceT = _source_tableau(doc, 'TCR')['texte']

    gA = CFG_V20['ACTIF_GROUPES'][RANG_PRECERT]
    cP = CFG_V20['PASSIF_COLS'][RANG_PRECERT]
    gT = CFG_V20['TCR_PAIRES'][RANG_PRECERT]

    wsA, wsP, wsT = wb[FEUILLE_ACTIF_V20], wb[FEUILLE_PASSIF_V20], wb[FEUILLE_TCR_V20]
    for ws, col, src in ((wsA, gA[0], sourceA), (wsP, cP, sourceP), (wsT, gT[0], sourceT)):
        ws[col+'1'] = ident.get('entreprise') or ''
        ws[col+'2'].number_format = '@'
        ws[col+'2'] = ident.get('nif_15') or ''
        ws[col+'3'] = src
    if ident.get('exercice_clos'):
        wsA[gA[0]+'7'] = ident['exercice_clos']

def _suspects_tcr_keys(doc):
    out = {}
    lecture = ((doc.get('controles') or {}).get('lecture_tcr') or {})
    for s in lecture.get('suspects_sens_debit_credit') or []:
        out[(s.get('poste'), s.get('exercice'))] = s
    return out

def _ecrire_saisie_v20(wb, doc):
    idx = _index_postes(doc)
    controles_excel = []
    ecrits = 0
    ignores = []
    suspects = _suspects_tcr_keys(doc)

    wsA, wsP, wsT = wb[FEUILLE_ACTIF_V20], wb[FEUILLE_PASSIF_V20], wb[FEUILLE_TCR_V20]
    gA = CFG_V20['ACTIF_GROUPES'][RANG_PRECERT]
    cP = CFG_V20['PASSIF_COLS'][RANG_PRECERT]
    gT = CFG_V20['TCR_PAIRES'][RANG_PRECERT]

    # ── ACTIF : brut + amort écrits ; net toujours calculé par le modèle
    for rc, rw in CFG_V20['ACTIF_ROWS'].items():
        vals = idx.get(('ACTIF', rc), {})
        mapping = [
            ('montant_brut', gA[0]),
            ('amortissements_provisions_pertes', gA[1]),
            ('net_n', gA[2]),
        ]
        for champ, col in mapping:
            v = vals.get(champ)
            if not isinstance(v, (int, float)) or isinstance(v, bool):
                continue
            cell = wsA[col + str(rw)]
            vk = _kdzd(v)
            if _est_formule(cell):
                controles_excel.append({
                    'type': 'CELLULE_FORMULE', 'tableau': 'ACTIF', 'poste': rc,
                    'champ': champ, 'feuille': FEUILLE_ACTIF_V20, 'cellule': cell.coordinate,
                    'valeur_extraite_kdzd': vk
                })
            else:
                cell.value = vk
                cell.number_format = '#,##0'
                ecrits += 1

    # ── PASSIF : rang N uniquement
    for rc, rw in CFG_V20['PASSIF_ROWS'].items():
        vals = idx.get(('PASSIF', rc), {})
        v = vals.get('n')
        if not isinstance(v, (int, float)) or isinstance(v, bool):
            continue
        cell = wsP[cP + str(rw)]
        vk = _kdzd(v)
        if _est_formule(cell):
            controles_excel.append({
                'type': 'CELLULE_FORMULE', 'tableau': 'PASSIF', 'poste': rc,
                'champ': 'n', 'feuille': FEUILLE_PASSIF_V20, 'cellule': cell.coordinate,
                'valeur_extraite_kdzd': vk
            })
        else:
            cell.value = vk
            cell.number_format = '#,##0'
            ecrits += 1

    # ── TCR : rang N uniquement ; aucune permutation Débit/Crédit
    for rc, rw in CFG_V20['TCR_ROWS'].items():
        vals = idx.get(('TCR', rc), {})
        d, c = vals.get('n_debit'), vals.get('n_credit')
        d_ok = isinstance(d, (int, float)) and not isinstance(d, bool)
        c_ok = isinstance(c, (int, float)) and not isinstance(c, bool)
        cellD, cellC = wsT[gT[0] + str(rw)], wsT[gT[1] + str(rw)]

        # Ligne calculée : contrôle sur le NET (Crédit - Débit), jamais d'écriture
        if _est_formule(cellD) or _est_formule(cellC) or rw in CFG_V20['TCR_FORMULES']:
            if d_ok or c_ok:
                extrait_net = _kdzd((float(c) if c_ok else 0.0) - (float(d) if d_ok else 0.0))
                controles_excel.append({
                    'type': 'TCR_NET_FORMULE', 'tableau': 'TCR', 'poste': rc,
                    'champ': 'NET', 'feuille': FEUILLE_TCR_V20,
                    'cellule_debit': cellD.coordinate, 'cellule_credit': cellC.coordinate,
                    'valeur_extraite_kdzd': extrait_net
                })
            continue

        # Sens inhabituel : conservé dans le JSON, mais non importé dans la feuille de saisie.
        suspect = suspects.get((rc, 'N'))
        if suspect:
            sens = suspect.get('sens_habituel')
            cible = cellC if sens == 'CREDIT' else cellD
            _bordure_couleur(cible, _ORANGE, 'medium')
            cible.fill = _ORANGE_FOND
            cible.comment = Comment(
                'LECTURE SUSPECTE QWEN — NON IMPORTEE\\n'
                'Poste : ' + rc + '\\n'
                'Valeur lue : ' + format(float(suspect.get('valeur_lue') or 0) / 1000.0, ',.3f') + ' KDZD\\n'
                'Lue en : ' + str(suspect.get('lu_en')) + '\\n'
                'Sens habituel : ' + str(sens) + '\\n'
                'Verifier le PDF. La valeur brute reste conservee dans le JSON et 0.Donnees extraites.',
                'Ya-Risk'
            )
            ignores.append({'tableau':'TCR', 'poste':rc, 'motif':'SENS_INHABITUEL_A_VERIFIER'})
            continue

        if d_ok:
            cellD.value = _kdzd(d); cellD.number_format = '#,##0'; ecrits += 1
        if c_ok:
            cellC.value = _kdzd(c); cellC.number_format = '#,##0'; ecrits += 1

    return ecrits, controles_excel, ignores

def _style_entete(ws, row, start_col, end_col):
    for c in ws.iter_cols(min_col=start_col, max_col=end_col, min_row=row, max_row=row):
        for cell in c:
            cell.font = Font(bold=True)
            cell.fill = PatternFill('solid', start_color='D9EAF7')
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

def _recreer_donnees_extraites(wb, doc):
    if FEUILLE_EXTRAIT in wb.sheetnames:
        del wb[FEUILLE_EXTRAIT]
    ws = wb.create_sheet(FEUILLE_EXTRAIT, 0)
    ident = doc.get('identite') or {}
    idx = _index_postes(doc)

    ws['A1'] = 'YA-RISK — DONNEES EXTRAITES QWEN — AVANT CERTIFICATION'
    ws['A1'].font = Font(bold=True, size=14)
    ws['A2'] = 'Entreprise'; ws['B2'] = ident.get('entreprise')
    ws['D2'] = 'NIF'; ws['E2'] = ident.get('nif_15') or ''
    ws['G2'] = 'Exercice'; ws['H2'] = ident.get('exercice_clos')
    ws['J2'] = 'Modele'; ws['K2'] = (doc.get('document') or {}).get('modele_extraction')
    ws['M2'] = 'Statut'; ws['N2'] = 'A_CERTIFIER'

    r = 4

    # ACTIF
    ws.cell(r,1,'ACTIF — montants DZD tels qu extraits du PDF'); ws.cell(r,1).font = Font(bold=True, size=12); r += 1
    headers = ['row_code','Libelle','Brut N (DZD)','Amort./Prov. N (DZD)','Net N (DZD)','Net N-1 (DZD)','Role','Source']
    for j,h in enumerate(headers,1): ws.cell(r,j,h)
    _style_entete(ws,r,1,len(headers)); r += 1
    srcA = _source_tableau(doc,'ACTIF')['texte']
    for rc, lib in SCHEMAS['ACTIF']['postes'].items():
        vals = idx.get(('ACTIF',rc),{})
        if not any(vals.get(c) is not None for c in SCHEMAS['ACTIF']['cols']):
            continue
        role = 'CONTROLE_FORMULE' if rc in {
            'immobilisations_corporelles','immobilisations_financieres','creances_emplois_assimiles',
            'disponibilites_assimiles','total_actif_non_courant','total_actif_courant','total_general_actif'
        } else 'SAISIE'
        row = [rc,lib,vals.get('montant_brut'),vals.get('amortissements_provisions_pertes'),
               vals.get('net_n'),vals.get('net_n1'),role,srcA]
        for j,v in enumerate(row,1): ws.cell(r,j,v)
        r += 1

    r += 1
    # PASSIF
    ws.cell(r,1,'PASSIF — montants DZD tels qu extraits du PDF'); ws.cell(r,1).font = Font(bold=True, size=12); r += 1
    headers = ['row_code','Libelle','N (DZD)','N-1 (DZD)','Role','Source']
    for j,h in enumerate(headers,1): ws.cell(r,j,h)
    _style_entete(ws,r,1,len(headers)); r += 1
    srcP = _source_tableau(doc,'PASSIF')['texte']
    for rc, lib in SCHEMAS['PASSIF']['postes'].items():
        vals = idx.get(('PASSIF',rc),{})
        if vals.get('n') is None and vals.get('n1') is None:
            continue
        role = 'CONTROLE_FORMULE' if rc in {
            'total_capitaux_propres','total_passifs_non_courants','total_passifs_courants','total_general_passif'
        } else 'SAISIE'
        row = [rc,lib,vals.get('n'),vals.get('n1'),role,srcP]
        for j,v in enumerate(row,1): ws.cell(r,j,v)
        r += 1

    r += 1
    # TCR
    ws.cell(r,1,'TCR — montants DZD tels qu extraits du PDF'); ws.cell(r,1).font = Font(bold=True, size=12); r += 1
    headers = ['row_code','Libelle','N Debit','N Credit','N-1 Debit','N-1 Credit','Mode','Sens habituel','Statut lecture','Source']
    for j,h in enumerate(headers,1): ws.cell(r,j,h)
    _style_entete(ws,r,1,len(headers)); r += 1
    srcT = _source_tableau(doc,'TCR')['texte']
    suspects = _suspects_tcr_keys(doc)
    for rc, lib in SCHEMAS['TCR']['postes'].items():
        vals = idx.get(('TCR',rc),{})
        if not any(vals.get(c) is not None for c in SCHEMAS['TCR']['cols']):
            continue
        sens = {'C':'CREDIT','D':'DEBIT','B':'DEBIT_OU_CREDIT'}.get(TCR_SENS.get(rc),'DEBIT_OU_CREDIT')
        statut = 'SENS_INHABITUEL_A_VERIFIER' if (rc,'N') in suspects or (rc,'N-1') in suspects else 'OK'
        row = [rc,lib,vals.get('n_debit'),vals.get('n_credit'),vals.get('n1_debit'),vals.get('n1_credit'),
               role_tcr(rc),sens,statut,srcT]
        for j,v in enumerate(row,1): ws.cell(r,j,v)
        if statut != 'OK':
            for j in range(1,len(headers)+1): ws.cell(r,j).fill = _ORANGE_FOND
        r += 1

    # Annexes — inventaire lisible, sans chercher à reproduire leur mise en page
    r += 2
    ws.cell(r,1,'ANNEXES / AUTRES DONNEES EXTRAITES'); ws.cell(r,1).font = Font(bold=True, size=12); r += 1
    headers = ['Tableau','row_code','Libelle','Colonne','Valeur','Page']
    for j,h in enumerate(headers,1): ws.cell(r,j,h)
    _style_entete(ws,r,1,len(headers)); r += 1
    for page, tab, bloc in iter_blocs(doc):
        if tab in ('ACTIF','PASSIF','TCR'):
            continue
        for rc, vals, lib in lire_bloc(bloc):
            for col, v in (vals or {}).items():
                if v is None:
                    continue
                row = [tab,rc,lib,col,v,page.get('numero_page_scannee')]
                for j,x in enumerate(row,1): ws.cell(r,j,x)
                r += 1

    ws.freeze_panes = 'A5'
    for col, width in {'A':30,'B':48,'C':20,'D':24,'E':20,'F':20,'G':22,'H':30,'I':28,'J':22}.items():
        ws.column_dimensions[col].width = width
    return ws

def _recreer_checklist(wb, doc, template_warnings=None, recalc_info=None):
    if FEUILLE_CHECK in wb.sheetnames:
        del wb[FEUILLE_CHECK]
    ws = wb.create_sheet(FEUILLE_CHECK, 1)
    ident = doc.get('identite') or {}
    ctrl = doc.get('controles') or {}
    synth = ctrl.get('synthese') or {}
    lecture = ctrl.get('lecture_tcr') or {}

    ws['A1'] = 'YA-RISK — CHECKLIST PRE-CERTIFICATION'
    ws['A1'].font = Font(bold=True, size=14)
    infos = [
        ('Statut','A_CERTIFIER'),
        ('Entreprise',ident.get('entreprise')),
        ('NIF',ident.get('nif_15')),
        ('Exercice',ident.get('exercice_clos')),
        ('Modele extraction',(doc.get('document') or {}).get('modele_extraction')),
        ('Identite homogene','OUI' if ident.get('dossier_homogene') else 'NON'),
        ('Formules en ecart',synth.get('formules_en_ecart')),
        ('Formules non verifiables',synth.get('formules_non_verifiables')),
        ('Controles en ecart',synth.get('controles_en_ecart')),
        ('Controles non verifiables',synth.get('controles_non_verifiables')),
        ('Lectures TCR suspectes',lecture.get('nb_suspects')),
        ('Totaux TCR non lus',lecture.get('nb_totaux_non_lus')),
        ('Arbitrage requis','OUI' if synth.get('arbitrage_requis') else 'NON'),
    ]
    r = 3
    for k,v in infos:
        ws.cell(r,1,k).font = Font(bold=True)
        ws.cell(r,2,v)
        r += 1

    def section(titre, headers, rows, fill=None):
        nonlocal r
        r += 1
        ws.cell(r,1,titre).font = Font(bold=True, size=12)
        r += 1
        for j,h in enumerate(headers,1): ws.cell(r,j,h)
        _style_entete(ws,r,1,len(headers)); r += 1
        if not rows:
            ws.cell(r,1,'Aucun')
            r += 1
            return
        for row in rows:
            for j,v in enumerate(row,1): ws.cell(r,j,v)
            if fill:
                for j in range(1,len(headers)+1): ws.cell(r,j).fill = fill
            r += 1

    section('ALERTES IDENTITE',
            ['Code','Gravite','Champ','Message'],
            [[a.get('code'),a.get('gravite'),a.get('champ'),a.get('message')] for a in ident.get('alertes',[])],
            _JAUNE_FOND)

    section('LECTURES TCR SUSPECTES — NON IMPORTEES DANS SAISIE TCR',
            ['Poste','Exercice','Valeur lue DZD','Lu en','Sens habituel','Statut'],
            [[s.get('poste'),s.get('exercice'),s.get('valeur_lue'),s.get('lu_en'),s.get('sens_habituel'),s.get('statut')]
             for s in lecture.get('suspects_sens_debit_credit',[])],
            _ORANGE_FOND)

    section('TOTAUX / SOLDES TCR NON LUS',
            ['Poste','Exercice','Statut','Commentaire'],
            [[s.get('poste'),s.get('exercice'),s.get('statut'),s.get('commentaire')]
             for s in lecture.get('totaux_non_lus',[])],
            _JAUNE_FOND)

    section('ECARTS FORMULES',
            ['Tableau','Poste','Colonne','Extrait DZD','Recalcule DZD','Ecart DZD','Statut'],
            [[f.get('tableau'),f.get('poste'),f.get('colonne'),f.get('valeur_extraite'),f.get('valeur_recalculee'),f.get('ecart'),f.get('statut')]
             for f in ctrl.get('formules',[]) if f.get('statut') == 'ecart_significatif'],
            _ROUGE_FOND)

    section('FORMULES NON VERIFIABLES',
            ['Tableau','Poste','Colonne','Motif','Composants manquants'],
            [[f.get('tableau'),f.get('poste'),f.get('colonne'),f.get('motif'),', '.join(f.get('postes_manquants') or [])]
             for f in ctrl.get('formules',[]) if f.get('statut') == 'non_verifiable'],
            _JAUNE_FOND)

    section('CONTROLES CROISES EN ECART / NON VERIFIABLES',
            ['Controle','Gravite','Valeur A','Valeur B','Ecart','Statut','Manquants'],
            [[c.get('controle'),c.get('gravite'),c.get('valeur_a'),c.get('valeur_b'),c.get('ecart'),c.get('statut'),
              ', '.join(c.get('postes_manquants') or [])]
             for c in ctrl.get('controles_croises',[]) if c.get('statut') in ('ecart_significatif','non_verifiable')],
            None)

    section('ANOMALIES DU MODELE EXCEL SOURCE',
            ['Formule / cellule'],
            [[x] for x in (template_warnings or [])],
            _JAUNE_FOND)

    if recalc_info:
        section('RECALCUL LIBREOFFICE',
                ['OK','Code','Message'],
                [[str(bool(recalc_info.get('ok'))),recalc_info.get('code'),recalc_info.get('message')]],
                _VERT_FOND if recalc_info.get('ok') else _JAUNE_FOND)

    ws.freeze_panes = 'A3'
    ws.column_dimensions['A'].width = 42
    ws.column_dimensions['B'].width = 22
    ws.column_dimensions['C'].width = 22
    ws.column_dimensions['D'].width = 48
    ws.column_dimensions['E'].width = 48
    ws.column_dimensions['F'].width = 22
    ws.column_dimensions['G'].width = 38
    return ws

def _comparer_formules_excel(path_xlsx, controles_excel, tolerance=TOLERANCE_EXCEL_KDZD):
    wb_f = openpyxl.load_workbook(path_xlsx)
    wb_v = openpyxl.load_workbook(path_xlsx, data_only=True)

    ecarts, arrondis, non_calc = [], [], []
    for ctrl in controles_excel:
        extrait = ctrl.get('valeur_extraite_kdzd')
        if extrait is None:
            continue
        feuille = ctrl['feuille']

        if ctrl['type'] == 'TCR_NET_FORMULE':
            d = wb_v[feuille][ctrl['cellule_debit']].value
            c = wb_v[feuille][ctrl['cellule_credit']].value
            if not isinstance(d, (int,float)) and not isinstance(c, (int,float)):
                non_calc.append(ctrl)
                continue
            calc = (float(d) if isinstance(d,(int,float)) else 0.0) + (float(c) if isinstance(c,(int,float)) else 0.0)
            cible_coord = ctrl['cellule_debit'] if extrait < 0 else ctrl['cellule_credit']
        else:
            cible_coord = ctrl['cellule']
            calc = wb_v[feuille][cible_coord].value
            if not isinstance(calc, (int,float)):
                non_calc.append(ctrl)
                continue
            calc = float(calc)

        ecart = calc - float(extrait)
        if abs(ecart) <= 1e-9:
            continue

        cell = wb_f[feuille][cible_coord]
        txt = (
            'CONTROLE YA-RISK — VALEUR PDF vs FORMULE\\n\\n'
            'Poste : ' + str(ctrl.get('poste')) + '\\n'
            'Montant extrait PDF : ' + format(float(extrait), ',.3f') + ' KDZD\\n'
            'Montant calcule Excel : ' + format(float(calc), ',.3f') + ' KDZD\\n'
            'Ecart (calcule - extrait) : ' + format(float(ecart), '+,.3f') + ' KDZD'
        )
        cell.comment = Comment(txt, 'Ya-Risk')

        rec = dict(ctrl)
        rec.update({'valeur_recalculee_kdzd': calc, 'ecart_kdzd': round(ecart,6), 'cellule_commentee': cible_coord})

        if abs(ecart) <= tolerance:
            cell.fill = _JAUNE_FOND
            _bordure_couleur(cell, 'C9A227', 'medium')
            rec['niveau'] = 'ARRONDI'
            arrondis.append(rec)
        elif abs(ecart) <= SEUIL_ECART_IMPORTANT_KDZD:
            cell.fill = _ORANGE_FOND
            _bordure_couleur(cell, _ORANGE, 'medium')
            rec['niveau'] = 'MOYEN'
            ecarts.append(rec)
        else:
            cell.fill = _ROUGE_FOND
            _bordure_couleur(cell, _ROUGE, 'thick')
            rec['niveau'] = 'IMPORTANT'
            ecarts.append(rec)

    wb_f.save(path_xlsx)
    return ecarts, arrondis, non_calc

def transposer_v20(doc, sortie_xlsx, modele=None):
    modele = Path(modele or MODELE_XLSX)
    sortie = Path(sortie_xlsx)
    tmp = sortie.with_suffix('.tmp.xlsx')
    sortie.parent.mkdir(parents=True, exist_ok=True)

    if not modele.exists():
        raise FileNotFoundError(
            'Modele analyste V20 introuvable : ' + str(modele) +
            '. Deposez MODEL ETAT FINANCIER - ANALYSTE COPILOTE V20.xlsx dans /mnt/Risk/Model.'
        )

    shutil.copy2(modele, tmp)
    wb = openpyxl.load_workbook(tmp)

    attendues = [FEUILLE_ACTIF_V20, FEUILLE_PASSIF_V20, FEUILLE_TCR_V20]
    absentes = [s for s in attendues if s not in wb.sheetnames]
    if absentes:
        raise ValueError('Modele V20 incompatible — feuilles absentes : ' + ', '.join(absentes))

    template_warnings = _scan_formules_invalides(wb)
    _preparer_zone_n(wb)
    _ecrire_identification_v20(wb, doc)
    ecrits, controles_excel, ignores = _ecrire_saisie_v20(wb, doc)
    _recreer_donnees_extraites(wb, doc)
    _recreer_checklist(wb, doc, template_warnings=template_warnings, recalc_info=None)

    # Demander un recalcul complet à l'ouverture, même si LibreOffice est absent.
    try:
        wb.calculation.fullCalcOnLoad = True
        wb.calculation.forceFullCalc = True
        wb.calculation.calcMode = 'auto'
    except Exception:
        pass
    wb.save(tmp)

    recalc1 = _recalculer_excel(tmp)
    ecarts, arrondis, non_calc = [], [], []
    if recalc1.get('ok'):
        ecarts, arrondis, non_calc = _comparer_formules_excel(tmp, controles_excel)
    else:
        non_calc = controles_excel

    # Checklist finale après comparaison
    wb2 = openpyxl.load_workbook(tmp)
    _recreer_checklist(wb2, doc, template_warnings=template_warnings, recalc_info=recalc1)
    wb2.save(sortie)
    tmp.unlink(missing_ok=True)

    # Second recalcul après ajout des commentaires/checklist
    recalc2 = _recalculer_excel(sortie)

    doc.setdefault('controles', {})['transposition'] = {
        'classeur': sortie.name,
        'modele_source': modele.name,
        'statut': 'A_CERTIFIER',
        'rang_alimente': RANG_PRECERT,
        'unite_json': 'DZD',
        'unite_excel': 'KDZD',
        'valeurs_ecrites': ecrits,
        'controles_cellules_formule': len(controles_excel),
        'ecarts_signales': len(ecarts),
        'ecarts_arrondis': len(arrondis),
        'formules_non_calculees': len(non_calc),
        'lectures_suspectes_non_importees': len(ignores),
        'detail_ecarts': ecarts,
        'detail_arrondis': arrondis,
        'detail_non_calcule': non_calc,
        'detail_non_importe': ignores,
        'anomalies_modele_source': template_warnings,
        'recalcul_initial': recalc1,
        'recalcul_final': recalc2
    }
    return sortie

print('✅ Pre-certification Excel V20 Ya-Risk V21 prete')
print('   Modele : ' + str(MODELE_XLSX))
print('   Recalc : ' + str(RECALC_TOOL) + ' | disponible=' + str(RECALC_DISPONIBLE))


In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 17 — EXECUTION POST-EXTRACTION | YARISK_V21
# Sorties :
#   - JSON avant certification
#   - Excel PRECERT basé sur le modèle analyste V20
# ════════════════════════════════════════════════════════════
CONTROLE_DIR = OUTPUT_DIR / 'json_avant_certification_v21'
XLSX_DIR = OUTPUT_DIR / 'excel_pre_certification_v21'
CONTROLE_DIR.mkdir(parents=True, exist_ok=True)
XLSX_DIR.mkdir(parents=True, exist_ok=True)

def traiter_dossier(chemin_json, avec_excel=True):
    doc = json.loads(Path(chemin_json).read_text(encoding='utf-8'))
    doc = appliquer_controles(doc)

    # Métadonnées de contrat avant certification
    doc['schema_version'] = '21.0'
    doc['workflow'] = {
        'statut': 'A_CERTIFIER',
        'certifie': False,
        'date_preparation': datetime.now().isoformat(timespec='seconds')
    }
    stem = Path(chemin_json).stem

    if avec_excel:
        if Path(MODELE_XLSX).exists():
            try:
                transposer_v20(doc, XLSX_DIR / ('PRECERT_' + stem + '.xlsx'))
            except Exception as e:
                log('❌ Transposition V20 ' + stem + ' : ' + type(e).__name__ + ': ' + str(e))
                doc.setdefault('controles', {})['transposition'] = {
                    'statut': 'ECHEC',
                    'erreur': type(e).__name__ + ': ' + str(e)
                }
        else:
            msg = 'Modele Excel analyste V20 introuvable : ' + str(MODELE_XLSX)
            log('⚠️ ' + msg)
            doc.setdefault('controles', {})['transposition'] = {'statut': 'NON_EXECUTEE', 'erreur': msg}

    sortie = CONTROLE_DIR / (stem + '.avant_certification.json')
    sortie.write_text(json.dumps(doc, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
    return doc, sortie

def resumer(doc):
    ident = doc.get('identite') or {}
    s = (doc.get('controles') or {}).get('synthese', {})
    lecture = (doc.get('controles') or {}).get('lecture_tcr', {})
    t = (doc.get('controles') or {}).get('transposition', {})

    print('  ' + str(ident.get('entreprise') or '(entreprise inconnue)') +
          ' | NIF ' + str(ident.get('nif_15') or '—') +
          ' | ' + str(ident.get('libelle_n') or 'N'))
    print('    Statut             : A_CERTIFIER')
    print('    Dossier homogene   : ' + ('OUI' if ident.get('dossier_homogene') else 'NON'))
    print('    Formules verifiees : ' + str(s.get('formules_verifiees')) +
          ' | ecarts=' + str(s.get('formules_en_ecart')) +
          ' | non verifiables=' + str(s.get('formules_non_verifiables')))
    print('    Controles croises  : ' + str(s.get('controles_verifies')) +
          ' | ecarts=' + str(s.get('controles_en_ecart')) +
          ' | non verifiables=' + str(s.get('controles_non_verifiables')))
    print('    TCR suspects       : ' + str(lecture.get('nb_suspects')) +
          ' | totaux non lus=' + str(lecture.get('nb_totaux_non_lus')))
    print('    Arbitrage requis   : ' + ('OUI' if s.get('arbitrage_requis') else 'NON'))

    if t:
        print('    Excel              : ' + str(t.get('statut')) +
              ' | valeurs ecrites=' + str(t.get('valeurs_ecrites')) +
              ' | ecarts=' + str(t.get('ecarts_signales')) +
              ' | arrondis=' + str(t.get('ecarts_arrondis')) +
              ' | suspects non importes=' + str(t.get('lectures_suspectes_non_importees')))
        if t.get('anomalies_modele_source'):
            print('    ⚠️ Modele source   : ' + str(len(t['anomalies_modele_source'])) +
                  ' formule(s) contiennent #REF!')

fichiers = sorted(JSON_DIR.glob('*.json'))
log('Post-traitement Ya-Risk V21 de ' + str(len(fichiers)) + ' dossier(s)')
for f in fichiers:
    try:
        doc, sortie = traiter_dossier(f)
        print()
        print('📄 ' + f.name)
        resumer(doc)
        print('    JSON avant certif  : ' + str(sortie))
    except Exception as e:
        log('❌ ' + f.name + ' : ' + type(e).__name__ + ': ' + str(e))

print()
print('✅ Post-traitement Ya-Risk V21 termine')
print('   JSON avant certification : ' + str(CONTROLE_DIR))
print('   Excel pre-certification  : ' + str(XLSX_DIR))


In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 18 — TESTS UNITAIRES LEGERS | YARISK_V21
# Aucun chargement de PDF ni modèle Excel requis.
# ════════════════════════════════════════════════════════════
_ok, _tot = 0, 0

def _t(libelle, obtenu, attendu):
    global _ok, _tot
    _tot += 1
    bon = obtenu == attendu
    _ok += int(bon)
    print(('✅' if bon else '❌') + ' ' + libelle +
          ('' if bon else ' | obtenu=' + repr(obtenu) + ' | attendu=' + repr(attendu)))

print('--- NIF strict ---')
_t('conforme', nif_15('002119116225582'), ('002119116225582', 'CONFORME'))
_t('14 chiffres non completes', nif_15('02119116225582'), (None, 'NON_CONFORME_14_CHIFFRES'))
_t('absent', nif_15(None), (None, 'ABSENT'))

print('--- TCR schema ---')
_t('charges personnel debit', TCR_SENS['charges_personnel'], 'D')
_t('produits financiers credit', TCR_SENS['produits_financiers'], 'C')
_t('resultat operationnel formule', role_tcr('resultat_operationnel'), 'CONTROLE_FORMULE')

print('--- Controle incomplet ---')
_idx = {
    ('ACTIF','a'): {'net_n': 100.0},
    ('ACTIF','b'): {'net_n': 50.0},
}
_s, _m = _somme_detail(_idx, [('+','ACTIF','a','net_n'),('+','ACTIF','b','net_n'),('+','ACTIF','c','net_n')], exiger_complet=True)
_t('somme partielle interdite', _s, None)
_t('composant manquant trace', _m, ['ACTIF/c/net_n'])

print('--- Detection sens TCR ---')
_doc_test = {
    'pages': [{
        'classification': {'types':['TCR'], 'page_blanche':False},
        'donnees': {
            'TCR': {
                'lignes': [
                    {'row_code':'produits_financiers','valeurs':{'n_debit':2000000,'n_credit':None,'n1_debit':None,'n1_credit':None}},
                    {'row_code':'resultat_operationnel','valeurs':{'n_debit':None,'n_credit':10000000,'n1_debit':None,'n1_credit':9000000}},
                ]
            }
        }
    }]
}
_lect = analyser_lecture_tcr(_doc_test)
_t('produit en debit signale', _lect['nb_suspects'], 1)

print()
print(str(_ok) + '/' + str(_tot) + ' tests passes')


---

# Partie 3 — Transposition multi-exercices V22 (N → N-4)

V21 remplit uniquement le **rang N**. Cette partie reprend le flux validé côté Copilot :
les JSON bruts V21 de **tous les PDF** sont convertis au format « exercice », fusionnés par
année (multi-PDF par exercice), puis injectés dans le V20 sur **5 rangs** avec :

- réinitialisation des cellules de saisie (formules jamais touchées) + retour à l'aspect initial ;
- conversion DZD → KDZD, NIF 15 chiffres en format texte, affichage `#,##0` ;
- identité (désignation / NIF / source PDF + pages) par feuille et par exercice ;
- **écarts > 1000 DZD** : bordure orange (moyen, ≤ 2000) / rouge épaisse (important) + commentaires ;
- **filet débit/crédit TCR** : valeur lue du mauvais côté = **non importée**, signalée (bordure orange + Checklist) ;
- **totaux non lus** signalés (un total est toujours imprimé : son absence = angle mort du contrôle) ;
- recoupement N-1 inter-PDF, équilibre Actif = Passif, cohérence NIF/désignation ;
- Checklist complète + masquage des colonnes des exercices non trouvés ;
- **workflow corrections** : `Baseline_JSON` / `Corrections_JSON` (mode validation en fin de notebook).

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 19 — MAPPING V20 MULTI-EXERCICES | YARISK_V22
# Identique au flux Copilot v3.8 (colonnes par rang, lignes, formules).
# ════════════════════════════════════════════════════════════
CFG = {
    "ACTIF_GROUPES": {"N": ["P", "Q", "R"], "N-1": ["M", "N", "O"], "N-2": ["J", "K", "L"],
                      "N-3": ["G", "H", "I"], "N-4": ["D", "E", "F"]},
    "PASSIF_COLS": {"N": "H", "N-1": "G", "N-2": "F", "N-3": "E", "N-4": "D"},
    "TCR_PAIRES": {"N": ["M", "N"], "N-1": ["K", "L"], "N-2": ["I", "J"],
                   "N-3": ["G", "H"], "N-4": ["E", "F"]},
    "ACTIF_ROWS": {"ecarts_acquisition_goodwill": 10, "immobilisations_incorporelles": 11,
                   "immobilisations_corporelles": 12, "terrains": 13, "batiments": 14,
                   "autres_immobilisations_corporelles": 15, "immobilisations_en_concession": 16,
                   "immobilisations_en_cours": 17, "immobilisations_financieres": 18,
                   "titres_mis_en_equivalence": 19, "autres_participations_creances": 20,
                   "autres_titres_immobilises": 21, "prets_actifs_financiers_non_courants": 22,
                   "impots_differes_actif": 23, "total_actif_non_courant": 24, "stocks_encours": 26,
                   "creances_emplois_assimiles": 27, "clients": 28, "autres_debiteurs": 29,
                   "impots_assimiles_actif": 30, "autres_creances_assimiles": 31,
                   "disponibilites_assimiles": 32, "placements_financiers_courants": 33,
                   "tresorerie_actif": 34, "total_actif_courant": 35, "total_general_actif": 36},
    "PASSIF_ROWS": {"capital_emis": 6, "capital_non_appele": 7, "primes_reserves": 8,
                    "ecart_reevaluation": 9, "ecart_equivalence": 10, "resultat_net_passif": 11,
                    "report_a_nouveau": 12, "part_societe_consolidante": 13, "part_minoritaires": 14,
                    "total_capitaux_propres": 15, "emprunts_dettes_financieres": 17,
                    "impots_differes_provisionnes": 18, "autres_dettes_non_courantes": 19,
                    "provisions_produits_avance": 20, "total_passifs_non_courants": 21,
                    "fournisseurs_rattaches": 23, "impots_passif": 24, "autres_dettes": 25,
                    "tresorerie_passif": 26, "total_passifs_courants": 27, "total_general_passif": 28},
    "TCR_ROWS": {"ventes_marchandises": 6, "produits_fabriques": 7, "prestations_services": 8,
                 "ventes_travaux": 9, "produits_annexes": 10, "rabais_remises_ristournes_accordes": 11,
                 "chiffre_affaires_net": 12, "production_stockee_destockee": 13,
                 "production_immobilisee": 14, "subvention_exploitation": 15, "production_exercice": 16,
                 "achats_marchandises_vendues": 17, "matieres_premieres": 18,
                 "autres_approvisionnements": 19, "variation_stocks": 20,
                 "achats_etudes_prestations": 21, "autres_consommations": 22,
                 "rabais_remises_ristournes_obtenus_achats": 23, "sous_traitance_generale": 24,
                 "locations": 25, "entretien_reparations": 26, "primes_assurances": 27,
                 "personnel_exterieur": 28, "remuneration_intermediaires": 29, "publicite": 30,
                 "deplacements_missions": 31, "autres_services": 32,
                 "rabais_remises_ristournes_obtenus_services": 33, "consommations_exercice": 34,
                 "valeur_ajoutee_exploitation": 35, "charges_personnel": 36,
                 "impots_taxes_assimiles": 37, "excedent_brut_exploitation": 38,
                 "autres_produits_operationnels": 39, "autres_charges_operationnelles": 40,
                 "dotations_amortissements": 41, "provisions": 42, "pertes_valeur": 43,
                 "reprises_pertes_valeur_provisions": 44, "resultat_operationnel": 45,
                 "produits_financiers": 46, "charges_financieres": 47, "resultat_financier": 48,
                 "resultat_ordinaire": 49, "elements_extraordinaires_produits": 50,
                 "elements_extraordinaires_charges": 51, "resultat_extraordinaire": 52,
                 "impots_exigibles_resultats": 53, "impots_differes_resultats": 54,
                 "resultat_net_exercice": 55},
    "ACTIF_FORMULES": [9, 12, 18, 24, 25, 27, 32, 35, 36],
    "PASSIF_FORMULES": [5, 15, 16, 21, 22, 27, 28],
    "TCR_FORMULES": [12, 16, 34, 35, 38, 45, 48, 49, 52, 55],
}

TCR_SENS = {"ventes_marchandises": "C", "produits_fabriques": "C", "prestations_services": "C",
            "ventes_travaux": "C", "produits_annexes": "C", "production_immobilisee": "C",
            "subvention_exploitation": "C", "rabais_remises_ristournes_obtenus_achats": "C",
            "rabais_remises_ristournes_obtenus_services": "C", "reprises_pertes_valeur_provisions": "C",
            "autres_produits_operationnels": "C", "produits_financiers": "C",
            "elements_extraordinaires_produits": "C", "rabais_remises_ristournes_accordes": "D",
            "achats_marchandises_vendues": "D", "matieres_premieres": "D",
            "autres_approvisionnements": "D", "achats_etudes_prestations": "D",
            "autres_consommations": "D", "sous_traitance_generale": "D", "locations": "D",
            "entretien_reparations": "D", "primes_assurances": "D", "personnel_exterieur": "D",
            "remuneration_intermediaires": "D", "publicite": "D", "deplacements_missions": "D",
            "autres_services": "D", "charges_personnel": "D", "impots_taxes_assimiles": "D",
            "autres_charges_operationnelles": "D", "dotations_amortissements": "D", "provisions": "D",
            "pertes_valeur": "D", "charges_financieres": "D", "elements_extraordinaires_charges": "D",
            "impots_exigibles_resultats": "D", "impots_differes_resultats": "D",
            "production_stockee_destockee": "B", "variation_stocks": "B", "chiffre_affaires_net": "B",
            "production_exercice": "B", "consommations_exercice": "B", "valeur_ajoutee_exploitation": "B",
            "excedent_brut_exploitation": "B", "resultat_operationnel": "B", "resultat_financier": "B",
            "resultat_ordinaire": "B", "resultat_extraordinaire": "B", "resultat_net_exercice": "B"}

TOTAUX_ATTENDUS = {
    "ACTIF": ["total_actif_non_courant", "total_actif_courant", "total_general_actif"],
    "PASSIF": ["total_capitaux_propres", "total_passifs_non_courants", "total_passifs_courants",
               "total_general_passif"],
    "TCR": ["chiffre_affaires_net", "production_exercice", "consommations_exercice",
            "valeur_ajoutee_exploitation", "excedent_brut_exploitation", "resultat_operationnel",
            "resultat_financier", "resultat_ordinaire", "resultat_extraordinaire",
            "resultat_net_exercice"],
}

TOTAUX_ENFANTS = {
    "ACTIF": {"total_actif_non_courant": ["ecarts_acquisition_goodwill", "immobilisations_incorporelles",
                                          "terrains", "batiments", "autres_immobilisations_corporelles",
                                          "immobilisations_en_concession", "immobilisations_en_cours",
                                          "titres_mis_en_equivalence", "autres_participations_creances",
                                          "autres_titres_immobilises", "prets_actifs_financiers_non_courants",
                                          "impots_differes_actif"],
              "total_actif_courant": ["stocks_encours", "clients", "autres_debiteurs",
                                      "impots_assimiles_actif", "autres_creances_assimiles",
                                      "placements_financiers_courants", "tresorerie_actif"]},
    "PASSIF": {"total_capitaux_propres": ["capital_emis", "primes_reserves", "ecart_reevaluation",
                                          "ecart_equivalence", "resultat_net_passif", "report_a_nouveau",
                                          "part_societe_consolidante", "part_minoritaires"],
               "total_passifs_non_courants": ["emprunts_dettes_financieres", "impots_differes_provisionnes",
                                              "autres_dettes_non_courantes", "provisions_produits_avance"],
               "total_passifs_courants": ["fournisseurs_rattaches", "impots_passif", "autres_dettes",
                                          "tresorerie_passif"]},
}

RANGS = ["N", "N-1", "N-2", "N-3", "N-4"]
TOLERANCE_KDZD = 1.0          # 1 KDZD = 1000 DZD
FEUILLE_CHECK = "Checklist"
FEUILLE_BASELINE = "Baseline_JSON"
FEUILLE_CORRECTIONS = "Corrections_JSON"
ROUGE, JAUNE, VERT = "FFC7CE", "FFF2CC", "C6EFCE"
ORANGE, ROUGEB, BLEU, ORANGECLAIR, BLEUCLAIR = "ED7D31", "FF0000", "0070C0", "F8CBAD", "DDEBF7"
log('Mapping V20 multi-exercices charge')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 20 — HELPERS EXCEL | YARISK_V22
# ════════════════════════════════════════════════════════════
def _bordure(cell, couleur, epaisseur):
    from openpyxl.styles import Border, Side
    s = Side(style=epaisseur, color="FF" + couleur)
    cell.border = Border(top=s, bottom=s, left=s, right=s)

def _sans_bordure(cell):
    from openpyxl.styles import Border
    cell.border = Border()

def _commentaire(cell, texte):
    from openpyxl.comments import Comment
    cell.comment = Comment(texte, "Import Liasses")

def _est_formule(cell):
    return isinstance(cell.value, str) and cell.value.startswith("=")

def _k(v):
    """DZD -> KDZD ; null reste null (jamais 0)."""
    return round(v / 1000, 3) if isinstance(v, (int, float)) else None

def _ecrire_json_long(ws, texte):
    """JSON long decoupe en cellules de 30000 caracteres (limite Excel 32767)."""
    ws.delete_rows(1, ws.max_row)
    for i in range(0, len(texte), 30000):
        ws.cell(row=i // 30000 + 1, column=1, value=texte[i:i + 30000])

def _lire_json_long(ws):
    morceaux = []
    for row in ws.iter_rows(min_col=1, max_col=1, max_row=20, values_only=True):
        if row[0]:
            morceaux.append(str(row[0]))
    return "".join(morceaux).strip()

def _info_tab(exo, code):
    """Normalise pages_trouvees : {pages, source_pdf} ou ancien format [n]."""
    entree = (exo.get("pages_trouvees") or {}).get(code)
    if not entree:
        return {"pages": [], "src": ""}
    if isinstance(entree, list):
        return {"pages": entree, "src": exo.get("source_pdf", "")}
    return {"pages": entree.get("pages", []),
            "src": entree.get("source_pdf") or exo.get("source_pdf", "")}

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 21 — ADAPTATEUR V21 -> EXERCICE + FUSION | YARISK_V22
# Convertit un doc V21 (pages/donnees) au format « exercice » du flux
# Copilot, puis fusionne par annee (multi-PDF par exercice).
# ════════════════════════════════════════════════════════════
def doc_v21_vers_exercice(doc):
    ident = doc.get('identite') or {}
    pages_t = {t: {'pages': [], 'source_pdf': ''} for t in ('ACTIF', 'PASSIF', 'TCR')}
    tableaux = {t: {'lignes': []} for t in ('ACTIF', 'PASSIF', 'TCR')}
    vus = set()
    for page in doc.get('pages', []):
        src = page.get('fichier_source') or ''
        n = page.get('numero_page_scannee')
        for tab, bloc in (page.get('donnees') or {}).items():
            if tab not in pages_t or not bloc:
                continue
            if isinstance(n, int):
                pages_t[tab]['pages'].append(n)
            if src and not pages_t[tab]['source_pdf']:
                pages_t[tab]['source_pdf'] = src
            for cle, vals, lib in lire_bloc(bloc):
                if str(cle).startswith('ligne_') or cle in ('total', ' TOTAL '):
                    continue
                if not any(v is not None for v in (vals or {}).values()):
                    continue
                if (tab, cle) in vus:
                    continue
                vus.add((tab, cle))
                tableaux[tab]['lignes'].append({'row_code': cle, 'libelle_imprime': lib, 'valeurs': vals})
    src_pdf = next((p.get('fichier_source') for p in doc.get('pages', []) if p.get('fichier_source')), '')
    return {'source_pdf': src_pdf, 'annee': ident.get('annee_n'),
            'exercice_clos': ident.get('exercice_clos'), 'nif': ident.get('nif_15') or '',
            'raison_sociale': ident.get('entreprise') or '', 'activite': ident.get('activite') or '',
            'pages_trouvees': pages_t, 'tableaux': tableaux}

def fusionner_exercices(tous):
    """Un seul objet par annee (multi-PDF par exercice)."""
    par_annee, doublons = {}, []
    for exo in tous:
        annee = exo.get('annee')
        if not annee:
            continue
        if annee not in par_annee:
            par_annee[annee] = exo
            continue
        base = par_annee[annee]
        for tab in ('ACTIF', 'PASSIF', 'TCR'):
            nouv = (exo.get('tableaux') or {}).get(tab, {}).get('lignes') or []
            if not nouv:
                continue
            exist = base.setdefault('tableaux', {}).setdefault(tab, {}).setdefault('lignes', [])
            connus = {l.get('row_code') for l in exist}
            ajoutes = 0
            for lig in nouv:
                if lig.get('row_code') not in connus:
                    exist.append(lig); ajoutes += 1
            if ajoutes == 0 and exist:
                doublons.append(f'{tab}/{annee}')
            pt = (exo.get('pages_trouvees') or {}).get(tab)
            if pt and pt.get('pages'):
                base.setdefault('pages_trouvees', {})[tab] = pt
    return par_annee, doublons

In [ ]:
import os, re, json, time, shutil, tempfile, subprocess
from pathlib import Path
from datetime import datetime
# ════════════════════════════════════════════════════════════
# CELLULE 22 — REMPLISSAGE V20 MULTI-EXERCICES | YARISK_V22
# Reset -> saisie -> identite -> corrections -> baseline -> controles -> Checklist.
# ════════════════════════════════════════════════════════════
def remplir_modele(par_annee, doublons, modele_path, sortie_path):
    import openpyxl
    wb = openpyxl.load_workbook(modele_path)
    wsA, wsP, wsT = wb["Saisie actif"], wb["Saisie passif"], wb["Saisie TCR"]

    annees = sorted(par_annee.keys(), reverse=True)[:5]
    rang_de = {a: RANGS[i] for i, a in enumerate(annees)}
    purges, ecrits = 0, 0
    suspects, ecarts, manquants, baseline = [], [], [], []
    totaux_pdf = {}   # (feuille, adresse) -> {attendu, poste, annee}

    # ── 2b. Reinitialisation : vide les cellules de saisie (JAMAIS les formules)
    #        + efface les bordures colorees (retour a l'aspect initial) ──
    def reinit(ws, cols, rows_map, formules, lignes_identite=(1, 2, 3), inclure_net=False):
        nonlocal purges
        toutes = sorted(set(rows_map.values()))
        for col in cols:
            for rw in range(1, max(toutes) + 1):
                if rw in toutes or rw in lignes_identite or rw == 7:
                    _sans_bordure(ws[f"{col}{rw}"])
            for rw in toutes:
                if rw in formules:
                    continue
                cell = ws[f"{col}{rw}"]
                if _est_formule(cell):
                    continue
                if cell.value not in (None, ""):
                    cell.value = None
                    purges += 1
            for rw in lignes_identite:
                cell = ws[f"{col}{rw}"]
                if not _est_formule(cell) and cell.value not in (None, ""):
                    cell.value = None
                    purges += 1

    for rang in RANGS:
        gA = CFG["ACTIF_GROUPES"][rang]
        reinit(wsA, gA[:2], CFG["ACTIF_ROWS"], CFG["ACTIF_FORMULES"], (1, 2, 3, 7))
        net_col = wsA[gA[2] + "1"].column_letter
        for rw in sorted(set(CFG["ACTIF_ROWS"].values())):      # net : formules preservees
            _sans_bordure(wsA[f"{net_col}{rw}"])
        reinit(wsP, [CFG["PASSIF_COLS"][rang]], CFG["PASSIF_ROWS"], CFG["PASSIF_FORMULES"])
        reinit(wsT, CFG["TCR_PAIRES"][rang], CFG["TCR_ROWS"], CFG["TCR_FORMULES"])
        for col in gA:
            wsA.column_dimensions[col].hidden = False
        wsP.column_dimensions[CFG["PASSIF_COLS"][rang]].hidden = False
        for col in CFG["TCR_PAIRES"][rang]:
            wsT.column_dimensions[col].hidden = False

    # ── 3. Remplissage par exercice ──
    def set_si_saisie(ws, addr, v):
        nonlocal ecrits
        if v is None:
            return
        cell = ws[addr]
        if _est_formule(cell):
            return
        cell.value = v
        cell.number_format = "#,##0"
        baseline.append({"f": ws.title, "a": addr, "v": v})
        ecrits += 1

    def garder(ws, addr, attendu, poste, annee):
        if attendu is not None:
            totaux_pdf[(ws.title, addr)] = {"attendu": attendu, "poste": poste, "annee": annee}

    for annee in annees:
        exo = par_annee[annee]
        rang = rang_de[annee]
        tabs = exo.get("tableaux") or {}
        gA = CFG["ACTIF_GROUPES"][rang]
        cB, cAm, cN = gA
        if exo.get("exercice_clos"):
            wsA[f"{cB}7"] = exo["exercice_clos"]
        for lig in (tabs.get("ACTIF", {}).get("lignes") or []):
            row = CFG["ACTIF_ROWS"].get(lig.get("row_code") or "")
            if not row:
                continue
            v = lig.get("valeurs") or {}
            brut, amo, net = _k(v.get("montant_brut")), _k(v.get("amortissements_provisions_pertes")), _k(v.get("net_n"))
            if row in CFG["ACTIF_FORMULES"]:
                garder(wsA, f"{cN}{row}", net, "ACTIF/" + str(lig.get("row_code")), annee)
            else:
                set_si_saisie(wsA, f"{cB}{row}", brut)
                set_si_saisie(wsA, f"{cAm}{row}", amo)
                garder(wsA, f"{cN}{row}", net, "ACTIF/" + str(lig.get("row_code")), annee)
        cP = CFG["PASSIF_COLS"][rang]
        for lig in (tabs.get("PASSIF", {}).get("lignes") or []):
            row = CFG["PASSIF_ROWS"].get(lig.get("row_code") or "")
            if not row:
                continue
            v = _k((lig.get("valeurs") or {}).get("n"))
            if row in CFG["PASSIF_FORMULES"]:
                garder(wsP, f"{cP}{row}", v, "PASSIF/" + str(lig.get("row_code")), annee)
            else:
                set_si_saisie(wsP, f"{cP}{row}", v)
        cD, cC = CFG["TCR_PAIRES"][rang]
        for lig in (tabs.get("TCR", {}).get("lignes") or []):
            row = CFG["TCR_ROWS"].get(lig.get("row_code") or "")
            if not row:
                continue
            v = lig.get("valeurs") or {}
            dV, cV = _k(v.get("n_debit")), _k(v.get("n_credit"))
            if row in CFG["TCR_FORMULES"]:
                garder(wsT, f"{cD}{row}", dV, "TCR/" + str(lig.get("row_code")) + " (D)", annee)
                garder(wsT, f"{cC}{row}", cV, "TCR/" + str(lig.get("row_code")) + " (C)", annee)
            else:
                # Filet de securite debit/credit : valeur du mauvais cote = NON importee, signalee
                sens = TCR_SENS.get(lig.get("row_code") or "", "B")
                if sens == "C" and dV is not None and cV is None:
                    suspects.append({"feuille": "Saisie TCR", "adresse": f"{cC}{row}",
                                     "poste": str(lig.get("row_code")), "annee": annee,
                                     "valeur": dV, "lu": "DEBIT", "attendu": "CREDIT"})
                elif sens == "D" and cV is not None and dV is None:
                    suspects.append({"feuille": "Saisie TCR", "adresse": f"{cD}{row}",
                                     "poste": str(lig.get("row_code")), "annee": annee,
                                     "valeur": cV, "lu": "CREDIT", "attendu": "DEBIT"})
                else:
                    set_si_saisie(wsT, f"{cD}{row}", dV)
                    set_si_saisie(wsT, f"{cC}{row}", cV)
        # Identite par exercice (lignes 1-3)
        nif = re.sub(r"\D", "", str(exo.get("nif") or ""))
        for ws, col, code in ((wsA, cB, "ACTIF"), (wsP, cP, "PASSIF"), (wsT, cD, "TCR")):
            it = _info_tab(exo, code)
            ws[f"{col}1"] = exo.get("raison_sociale") or ""
            c_nif = ws[f"{col}2"]
            c_nif.number_format = "@"          # TEXTE : conserve les 15 chiffres
            c_nif.value = nif
            if not re.fullmatch(r"[0-9]{15}", nif):
                _commentaire(c_nif, f"NIF NON CONFORME [{annee}] : « {exo.get('nif') or '?'} » — "
                                    "15 chiffres attendus. Verifiez la page d'identification du PDF.")
            ws[f"{col}3"] = it["src"] + (" — p." + ",".join(map(str, it["pages"])) if it["pages"] else "")

    # ── 3b. Reapplication des corrections validees (onglet Corrections_JSON) ──
    nb_corr = 0
    if FEUILLE_CORRECTIONS in wb.sheetnames:
        try:
            doc_c = json.loads(_lire_json_long(wb[FEUILLE_CORRECTIONS]) or "{}")
            for co in doc_c.get("corrections", []):
                if co.get("f") not in wb.sheetnames:
                    continue
                cell = wb[co["f"]][co["a"]]
                if _est_formule(cell):
                    continue
                cell.value = co["apres"]
                cell.number_format = "#,##0"
                _bordure(cell, BLEU, "medium")
                _commentaire(cell, f"CORRECTION VALIDEE — import PDF : {round(co['avant'])} KDZD "
                                   f"-> corrige : {round(co['apres'])} KDZD")
                nb_corr += 1
        except Exception:
            pass

    # ── 3c. Baseline (reference PDF pour le mode validation) ──
    if FEUILLE_BASELINE not in wb.sheetnames:
        wb.create_sheet(FEUILLE_BASELINE)
    _ecrire_json_long(wb[FEUILLE_BASELINE], json.dumps({"cellules": baseline}, ensure_ascii=False))

    # ── 4. Recoupement inter-PDF : N-1 du PDF Y+1 vs N du PDF Y ──
    recoup = []
    for i in range(len(annees) - 1):
        y_new, y_old = annees[i], annees[i + 1]
        e_new, e_old = par_annee[y_new], par_annee[y_old]
        nb = 0
        for tab, col_new, col_old in (("ACTIF", "net_n1", "net_n"), ("PASSIF", "n1", "n"),
                                      ("TCR", "n1_debit", "n_debit"), ("TCR", "n1_credit", "n_credit")):
            m_new, m_old = {}, {}
            for l in (e_new.get("tableaux") or {}).get(tab, {}).get("lignes") or []:
                v = (l.get("valeurs") or {}).get(col_new)
                if isinstance(v, (int, float)) and l.get("row_code"):
                    m_new[l["row_code"]] = v
            for l in (e_old.get("tableaux") or {}).get(tab, {}).get("lignes") or []:
                v = (l.get("valeurs") or {}).get(col_old)
                if isinstance(v, (int, float)) and l.get("row_code"):
                    m_old[l["row_code"]] = v
            nb += sum(1 for rc, vn in m_new.items()
                      if rc in m_old and abs(vn - m_old[rc]) > 1000)
        recoup.append(f"{y_old} : " + ("OK" if nb == 0 else f"{nb} poste(s) divergent(s) entre les 2 PDF"))

    # ── 5. Controle des totaux : recalcul LibreOffice, sinon repli Python ──
    calcule = recalculer_libreoffice(wb, sortie_path)
    nb_ano, nb_arr = 0, 0
    for (feuille, addr), info in totaux_pdf.items():
        raw = None
        if calcule is not None and feuille in calcule.sheetnames:
            raw = calcule[feuille][addr].value
        if not isinstance(raw, (int, float)):
            raw = totaux_python(par_annee, info, feuille)   # repli approche (actif/passif)
        if not isinstance(raw, (int, float)):
            continue
        ecart = round(raw - info["attendu"], 3)
        if ecart == 0 or abs(ecart) <= TOLERANCE_KDZD:
            if ecart != 0:
                nb_arr += 1
            continue
        grave = abs(ecart) > 2          # > 2000 DZD = IMPORTANT ; 1000-2000 = MOYEN
        cell = wb[feuille][addr]
        _bordure(cell, ROUGEB if grave else ORANGE, "thick" if grave else "medium")
        _commentaire(cell, f"{'ECART IMPORTANT' if grave else 'ECART MOYEN'} [{info['annee']}] {info['poste']}"
                           f" — PDF : {round(info['attendu'])} KDZD | Recalcule : {round(raw)} KDZD | "
                           f"Ecart : {'+' if ecart > 0 else ''}{round(ecart)} KDZD")
        nb_ano += 1
        ecarts.append({"feuille": feuille, "adresse": addr, "poste": info["poste"], "annee": info["annee"],
                       "attendu": round(info["attendu"]), "calcule": round(raw), "ecart": round(ecart),
                       "type": "ECART IMPORTANT" if grave else "ECART MOYEN"})

    # ── 6. Equilibre Actif = Passif + coherence identite + suspects + totaux non lus ──
    equil = []
    for annee in annees:
        rang = rang_de[annee]
        colA, colP = CFG["ACTIF_GROUPES"][rang][2], CFG["PASSIF_COLS"][rang]
        ta = _valeur_calculee(calcule, "Saisie actif", f"{colA}36")
        tp = _valeur_calculee(calcule, "Saisie passif", f"{colP}28")
        if isinstance(ta, (int, float)) and isinstance(tp, (int, float)):
            e = round(ta - tp, 3)
            equil.append(f"{annee} : " + ("OK" if abs(e) <= TOLERANCE_KDZD else f"ECART {'+' if e > 0 else ''}{e} KDZD"))
            if abs(e) > TOLERANCE_KDZD:
                c = wsP[f"{colP}28"]
                _bordure(c, ROUGEB, "thick")
                _commentaire(c, f"DESEQUILIBRE {annee} : Actif {round(ta)} ≠ Passif {round(tp)} KDZD")
                ecarts.append({"feuille": "Saisie passif", "adresse": f"{colP}28",
                               "poste": "total_general_passif", "annee": annee, "attendu": round(ta),
                               "calcule": round(tp), "ecart": round(e), "type": "DESEQUILIBRE"})
        else:
            equil.append(f"{annee} : non verifiable")

    nifs = {re.sub(r"\D", "", str(par_annee[a].get("nif") or "")) for a in annees} - {""}
    rs = {(par_annee[a].get("raison_sociale") or "").strip().lower() for a in annees} - {""}
    ident_ok = len(nifs) <= 1 and len(rs) <= 1
    ident_msg = ("OK — meme entreprise sur tous les exercices" if ident_ok else
                 f"ALERTE : {len(nifs)} NIF / {len(rs)} designations differents — verifiez que tous "
                 "les PDF appartiennent a la meme entreprise")

    for sp in suspects:
        cible = wsT[sp["adresse"]]
        _bordure(cible, ORANGE, "thick")
        _commentaire(cible, f"LECTURE SUSPECTE [{sp['annee']}] {sp['poste']} : {round(sp['valeur'])} KDZD "
                            f"lus en colonne {sp['lu']} (nature habituelle : {sp['attendu']}). NON IMPORTE — "
                            "verifiez le PDF ; si la lecture est confirmee, saisissez manuellement puis "
                            "lancez le mode validation.")

    for annee in annees:
        exo = par_annee[annee]
        for tab in ("ACTIF", "PASSIF", "TCR"):
            ligs = (exo.get("tableaux") or {}).get(tab, {}).get("lignes") or []
            if not ligs:
                continue
            trouves = set()
            for l in ligs:
                if l.get("row_code") and any(isinstance(v, (int, float))
                                             for v in (l.get("valeurs") or {}).values()):
                    trouves.add(l["row_code"])
            for rc in TOTAUX_ATTENDUS[tab]:
                if rc not in trouves:
                    manquants.append({"tab": tab, "poste": rc, "annee": annee})

    # ── 7. Checklist ──
    ecrire_checklist(wb, par_annee, annees, rang_de, recoup, equil, ecarts, suspects,
                     manquants, doublons, ident_ok, ident_msg, calcule is not None)

    # ── 8. Masquer les exercices non trouves ──
    for i in range(len(annees), 5):
        r = RANGS[i]
        for col in CFG["ACTIF_GROUPES"][r]:
            wsA.column_dimensions[col].hidden = True
        wsP.column_dimensions[CFG["PASSIF_COLS"][r]].hidden = True
        for col in CFG["TCR_PAIRES"][r]:
            wsT.column_dimensions[col].hidden = True

    wb.save(sortie_path)
    resume = (f"Exercices importes : {', '.join(f'{rang_de[a]}={a}' for a in annees)}"
              f" | Cellules reinitialisees : {purges} | Cellules ecrites : {ecrits}"
              f" | Ecarts > 1000 DZD : {nb_ano}" + (f" | ecarts <= 1000 DZD masques : {nb_arr}" if nb_arr else "")
              + f" | Identite : {'OK' if ident_ok else 'ALERTE'}"
              + (f" | Corrections validees reappliquees : {nb_corr}" if nb_corr else "")
              + (f" | LECTURES SUSPECTES (non importees) : {len(suspects)}" if suspects else "")
              + (f" | TOTAUX NON LUS (controle incomplet) : {len(manquants)}" if manquants else "")
              + (f" | DOUBLONS : {','.join(doublons)}" if doublons else "")
              + ("" if calcule is not None else " | Recalcul LibreOffice absent : ecarts actif/passif approches, TCR non controle"))
    print(resume)
    return resume

In [ ]:
import os, re, json, time, shutil, tempfile, subprocess
from pathlib import Path
from datetime import datetime
# ════════════════════════════════════════════════════════════
# CELLULE 23 — RECALCUL + CHECKLIST | YARISK_V22
# ════════════════════════════════════════════════════════════
def recalculer_libreoffice(wb, sortie_path):
    """Sauvegarde temporaire puis recalcul via LibreOffice (openpyxl ne calcule pas).
    Retourne un workbook data_only avec les valeurs recalculees, ou None."""
    soffice = shutil.which("soffice") or shutil.which("libreoffice")
    if not soffice:
        return None
    tmp_in = tempfile.mkdtemp()
    tmp_out = tempfile.mkdtemp()
    try:
        src = os.path.join(tmp_in, "recalc.xlsx")
        wb.save(src)          # openpyxl n'ecrit pas de cache : LO recalcule tout au chargement
        subprocess.run([soffice, "--headless", "--convert-to", "xlsx", "--outdir", tmp_out, src],
                       check=True, timeout=600, capture_output=True)
        import openpyxl
        return openpyxl.load_workbook(os.path.join(tmp_out, "recalc.xlsx"), data_only=True)
    except Exception as exc:
        print(f"  [Recalcul] LibreOffice indisponible : {exc}")
        return None
    finally:
        shutil.rmtree(tmp_in, ignore_errors=True)
        shutil.rmtree(tmp_out, ignore_errors=True)

def _valeur_calculee(calcule, feuille, addr):
    if calcule is not None and feuille in calcule.sheetnames:
        return calcule[feuille][addr].value
    return None

def totaux_python(par_annee, info, feuille):
    """Repli approche : somme des lignes enfants importees (actif/passif uniquement).
    Le TCR n'est pas controle en mode repli (soldes trop specifiques au modele)."""
    m = re.match(r"(ACTIF|PASSIF)/([a-z_]+)", info["poste"])
    if not m:
        return None
    tab, rc = m.group(1), m.group(2)
    enfants = TOTAUX_ENFANTS.get(tab, {}).get(rc)
    if not enfants:
        return None
    exo = par_annee.get(info["annee"])
    if not exo:
        return None
    col = "net_n" if tab == "ACTIF" else "n"
    vals = {}
    for l in (exo.get("tableaux") or {}).get(tab, {}).get("lignes") or []:
        v = (l.get("valeurs") or {}).get(col)
        if isinstance(v, (int, float)) and l.get("row_code"):
            vals[l["row_code"]] = v / 1000.0
    if not any(e in vals for e in enfants):
        return None
    return round(sum(vals.get(e, 0.0) for e in enfants), 3)

def ecrire_checklist(wb, par_annee, annees, rang_de, recoup, equil, ecarts, suspects,
                     manquants, doublons, ident_ok, ident_msg, recalcul_ok):
    from openpyxl.styles import PatternFill, Font
    if FEUILLE_CHECK in wb.sheetnames:
        del wb[FEUILLE_CHECK]
    ws = wb.create_sheet(FEUILLE_CHECK)
    e0 = par_annee[annees[0]] if annees else {}
    lignes = [
        ["DOSSIER", e0.get("raison_sociale") or "?", "", "", "", "", "", "", "", "", ""],
        ["NIF", e0.get("nif") or "?", "Activite", e0.get("activite") or "?", "", "", "", "", "", "", ""],
        ["CONTROLE IDENTITE", ident_msg, "", "", "", "", "", "", "", "", ""],
        ["Pipeline", "Domino + Qwen v1.0 — extraction LLM + controles deterministes (null jamais traite comme 0)", "", "", "", "", "", "", "", "", ""],
        ["Exercice", "Rang", "Source PDF", "Cloture", "ACTIF", "PASSIF", "TCR",
         "Recoupement N-1 inter-PDF", "Equilibre A=P", "Ecarts", "Date import"],
    ]
    from datetime import datetime
    now = datetime.now().strftime("%d/%m/%Y %H:%M")

    def fmt(code, exo):
        it = _info_tab(exo, code)
        nb = len((exo.get("tableaux") or {}).get(code, {}).get("lignes") or [])
        if not it["pages"] and nb == 0:
            return "KO — non trouve"
        morceaux = []
        if it["pages"]:
            morceaux.append("p." + ",".join(map(str, it["pages"])))
        if it["src"]:
            morceaux.append(it["src"])
        morceaux.append(f"{nb} lignes")
        return ("PARTIEL (" if 0 < nb < 8 else "OK (") + " — ".join(morceaux) + ")" + \
               (" — verifier le PDF (scan ?)" if 0 < nb < 8 else "")

    for annee in annees:
        exo = par_annee[annee]
        srcs = []
        for code in ("ACTIF", "PASSIF", "TCR"):
            s = _info_tab(exo, code)["src"]
            if s and s not in srcs:
                srcs.append(s)
        if not srcs and exo.get("source_pdf"):
            srcs.append(exo["source_pdf"])
        rec = next((r for r in recoup if r.startswith(str(annee))), "—")
        eq = next((r for r in equil if r.startswith(str(annee))), "—")
        nb_ec = sum(1 for ec in ecarts if ec["annee"] == annee)
        lignes.append([annee, rang_de.get(annee, "-"), " + ".join(srcs) if srcs else "?",
                       exo.get("exercice_clos") or "?", fmt("ACTIF", exo), fmt("PASSIF", exo),
                       fmt("TCR", exo), rec, eq, nb_ec, now])
    for d in doublons:
        lignes.append([d, "DOUBLON", "2 PDF portent le meme exercice — le 1er a ete retenu",
                       "", "", "", "", "", "", "", now])
    for i in range(len(annees), 5):
        lignes.append(["—", f"{RANGS[i]} (non trouve)", "", "", "KO", "KO", "KO", "", "", "", now])
    data_start, data_end = 5, len(lignes)

    lignes.append([""] * 11)
    lignes.append(["ECARTS DETECTES (> 1000 DZD)",
                   "(commentaires sur les cellules + cette liste"
                   + ("" if recalcul_ok else " — recalcul LibreOffice absent : controle approche actif/passif, TCR non controle")
                   + ")", "", "", "", "", "", "", "", "", ""])
    lignes.append(["Feuille", "Cellule", "Poste", "Exercice", "PDF (KDZD)", "Recalcule (KDZD)",
                   "Ecart (KDZD)", "Type", "", "", ""])
    ecarts_header = len(lignes) - 1
    if not ecarts:
        lignes.append(["Aucun ecart superieur a 1000 DZD"] + [""] * 10)
    for ec in ecarts:
        lignes.append([ec["feuille"], ec["adresse"], ec["poste"], ec["annee"], ec["attendu"],
                       ec["calcule"], ec["ecart"], ec["type"], "", "", ""])

    lignes.append([""] * 11)
    lignes.append(["LECTURES SUSPECTES (sens debit/credit) — NON IMPORTEES",
                   "(valeur lue dans la mauvaise colonne — verifiez le PDF et saisissez "
                   "manuellement si confirme, puis lancez le mode validation)", "", "", "", "", "", "", "", "", ""])
    lignes.append(["Feuille", "Cellule cible", "Poste", "Exercice", "Valeur lue (KDZD)",
                   "Lu en", "Nature attendue", "", "", "", ""])
    suspects_header = len(lignes) - 1
    if not suspects:
        lignes.append(["Aucune lecture suspecte"] + [""] * 10)
    for sp in suspects:
        lignes.append([sp["feuille"], sp["adresse"], sp["poste"], sp["annee"], round(sp["valeur"]),
                       sp["lu"], sp["attendu"], "", "", "", ""])

    lignes.append([""] * 11)
    lignes.append(["TOTAUX NON LUS DANS LE JSON (controle des ecarts impossible sur ces lignes)",
                   "(un total est TOUJOURS imprime sur le PDF : s'il est absent, relancez "
                   "l'extraction sur ce tableau)", "", "", "", "", "", "", "", "", ""])
    lignes.append(["Tableau", "Poste (total attendu)", "Exercice"] + [""] * 8)
    manquants_header = len(lignes) - 1
    if not manquants:
        lignes.append(["Aucun — tous les totaux attendus sont presents"] + [""] * 10)
    for mq in manquants:
        lignes.append([mq["tab"], mq["poste"], mq["annee"]] + [""] * 8)

    for r, ligne in enumerate(lignes, start=1):
        for c, val in enumerate(ligne, start=1):
            ws.cell(row=r, column=c, value=val)
    bold = Font(bold=True)
    ws["A1"].font = bold
    for c in range(1, 12):
        ws.cell(row=5, column=c).font = bold
    if not ident_ok:
        for c in range(1, 12):
            ws.cell(row=3, column=c).fill = PatternFill("solid", fgColor=ROUGE)
    for r in range(data_start + 1, data_end + 1):
        for c in (5, 6, 7):
            cell = ws.cell(row=r, column=c)
            t = str(cell.value or "")
            cell.fill = PatternFill("solid", fgColor=VERT if t.startswith("OK")
                                    else JAUNE if t.startswith("PARTIEL") else ROUGE)
    for hdr, larg in ((ecarts_header, 8), (suspects_header, 7), (manquants_header, 3)):
        for c in range(1, larg + 1):
            ws.cell(row=hdr + 1, column=c).font = bold
    for r in range(ecarts_header + 2, len(lignes) + 1):
        t = str(ws.cell(row=r, column=8).value or "")
        if t in ("ECART IMPORTANT", "DESEQUILIBRE"):
            for c in range(1, 9):
                ws.cell(row=r, column=c).fill = PatternFill("solid", fgColor=ROUGE)
        elif t == "ECART MOYEN":
            for c in range(1, 9):
                ws.cell(row=r, column=c).fill = PatternFill("solid", fgColor=ORANGECLAIR)
    if suspects:
        for r in range(suspects_header + 2, suspects_header + 2 + len(suspects)):
            for c in range(1, 8):
                ws.cell(row=r, column=c).fill = PatternFill("solid", fgColor=ORANGECLAIR)
    if manquants:
        for r in range(manquants_header + 2, manquants_header + 2 + len(manquants)):
            for c in range(1, 4):
                ws.cell(row=r, column=c).fill = PatternFill("solid", fgColor=JAUNE)
    for col, larg in zip("ABCDEFGHIJK", (26, 14, 40, 12, 34, 34, 34, 30, 16, 8, 16)):
        ws.column_dimensions[col].width = larg

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 24 — EXECUTION MULTI-EXERCICES | YARISK_V22
# Charge tous les JSON bruts V21, transpose sur 5 rangs dans le V20.
# ════════════════════════════════════════════════════════════
json_bruts = sorted(JSON_DIR.glob('*.json'))
log(f'{len(json_bruts)} JSON brut(s) V21 trouves dans {JSON_DIR}')
tous_exos = []
for jp in json_bruts:
    try:
        tous_exos.append(doc_v21_vers_exercice(json.load(open(jp, encoding='utf-8'))))
    except Exception as exc:
        log(f'[ADAPTATEUR] {jp.name} ignore : {exc}')
par_annee, doublons = fusionner_exercices(tous_exos)
if not par_annee:
    log('Aucun exercice exploitable — lancez d abord l extraction V21 (cellules 1-11).')
else:
    annees = sorted(par_annee, reverse=True)[:5]
    sortie = OUTPUT_DIR / f"MODELE_V20_REMPLI_{annees[0]}.xlsx"
    resume = remplir_modele(par_annee, doublons, str(MODELE_XLSX), str(sortie))
    log('Fichier produit : ' + str(sortie))

In [ ]:
import os, re, json, time, shutil, tempfile, subprocess
from pathlib import Path
from datetime import datetime
# ════════════════════════════════════════════════════════════
# CELLULE 25 — VALIDATION DES CORRECTIONS (a la demande) | YARISK_V22
# Apres corrections manuelles dans le fichier produit : enregistre les
# ecarts vs baseline dans Corrections_JSON (bordure bleue), reappliquees
# automatiquement au prochain import.
# ════════════════════════════════════════════════════════════
def validation_corrections(path):
    """Compare les valeurs actuelles a Baseline_JSON ; les differences deviennent des
    corrections tracees (bordure bleue + commentaire) dans Corrections_JSON."""
    import openpyxl
    wb = openpyxl.load_workbook(path)
    if FEUILLE_BASELINE not in wb.sheetnames:
        print("Pas d'onglet Baseline_JSON : lancez d'abord un import.")
        return
    base = json.loads(_lire_json_long(wb[FEUILLE_BASELINE]) or "{}").get("cellules", [])
    corrections = []
    for b in base:
        if b["f"] not in wb.sheetnames:
            continue
        cell = wb[b["f"]][b["a"]]
        v = cell.value
        if isinstance(v, (int, float)) and abs(v - b["v"]) > 0.0001:
            corrections.append({"f": b["f"], "a": b["a"], "avant": b["v"], "apres": v})
            _bordure(cell, BLEU, "medium")
            _commentaire(cell, f"CORRECTION MANUELLE — import PDF : {round(b['v'])} KDZD "
                               f"-> corrige : {round(v)} KDZD")
    if FEUILLE_CORRECTIONS not in wb.sheetnames:
        wb.create_sheet(FEUILLE_CORRECTIONS)
    _ecrire_json_long(wb[FEUILLE_CORRECTIONS],
                      json.dumps({"corrections": corrections}, ensure_ascii=False))
    wb.save(path)
    print(f"Validation terminee : {len(corrections)} correction(s) enregistree(s) dans "
          f"{FEUILLE_CORRECTIONS} (elles seront reappliquees au prochain import).")

# FICHIER_A_VALIDER = OUTPUT_DIR / 'MODELE_V20_REMPLI_2024.xlsx'
# validation_corrections(str(FICHIER_A_VALIDER))